In [ ]:
### Importing required libraries ###

import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)  # Add this line
import seaborn as sns
from linearmodels.panel import PanelOLS
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import scipy
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm
from linearmodels import PooledOLS
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import itertools
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning
warnings.filterwarnings("ignore", category=ValueWarning)
warnings.filterwarnings(
    "ignore",
    message="Non-invertible starting MA parameters found"
)

# Table of contents

1. Context  
2. Functions  
2. Exploratory analysis  
    2.1. Read data  
    2.2. Analysis of missing values  
    2.3. Analysis of outliers  
    2.4. Descriptive statistics  
    2.5. Inconsistencies/ Data Transformation  
3. Cleaning of data  
    3.1. Treating of missing values/ outliers/ inconsistencies  
    3.2. Filter of data  
    3.3. Merge of data  
4. Development of recycling  
    4.1. How has the amount of recycling of waste developed in Europe over time?  
    4.2. How does recycling compare across countries?     
    4.3. How does it compare for specific types of waste?   
5. Are there characteristics of countries that could lead to increased recycling?   
6. How well can the development of the amount of recycling be predicted?  
7. Findings  
    7.1. Potential bias  
    7.2. Why were the questions good?  
    7.3. Do the answers make any sense?  
    7.4. Were there any difficulties?  
    7.5. Which Data Science tools and techniques were learned during this exercise?  
    7.6. Key insights?  
8. Appendix  
    8.1. Usage of LLM  
    8.2. Distribution of work  
    8.3. References  

# 1. Context

## EU Recycling
European policymakers have long recognized that better waste recycling is crucial for environmental protection, resource conservation, and sustainable growth. EU waste policy is central to the Circular Economy vision – the idea of keeping materials in use for as long as possible and minimizing waste. In fact, EU directives on waste management have been enacted to push member states up the “waste hierarchy” (prioritizing prevention, then reuse, then recycling, over disposal [3]). These laws aim to extract value from what we throw away and reduce environmental harm. For example, the Waste Framework Directive (2008/98/EC) created a broad legal framework for waste in Europe, setting ambitious recycling targets and mandating national waste plans. Under this directive, countries were required to recycle at least 50% of household (municipal) waste by 2020, with even higher goals of 60% by 2030 and 65% by 2035 [1]. Complementing this, the Packaging and Packaging Waste Directive (94/62/EC, later updated) compels high recovery of packaging materials (aiming for about 65% recycling of all packaging by 2025 [1]), and the WEEE Directive (2012/19/EU) addresses electronic waste by setting collection and recycling requirements for e-waste. Through such policies – along with others like the Landfill Directive to curb dumping – the EU has built a comprehensive regulatory push to improve recycling rates and move Europe toward a circular economy. The motivation behind these directives is clear: waste that isn’t recycled represents lost raw materials and often ends up polluting land, water, and air. By enforcing common rules, the EU strives to ensure all member states contribute to the environmental goals and benefit from the economic opportunities of recycling (like job creation in the recycling industry and reduced raw material imports) [3].
 
## Recycling Disparities
Despite these regulations and goals, monitoring compliance and progress is a critical ongoing need. Europe’s waste challenge remains significant – the average European generates around 500–600 kg of municipal waste per year, and as of 2020 roughly 49% of that (just about half) was recycled [1] [2]. This is a notable improvement from a decade ago (when only around 37% was recycled [1]), indicating that EU policies and national efforts have had an impact. However, this EU-wide average hides stark differences across countries. In some member states, less than a third of waste gets recycled, and over 60% of household waste still ends up in landfills [3] – a sign of lagging performance – whereas other states have become recycling frontrunners, already surpassing the EU’s 2025 targets with well over half of their waste recycled. These disparities raise important questions about why certain countries succeed and others struggle, and they underscore why the EU cares deeply about tracking and understanding recycling progress. The European Commission doesn’t just set targets – it actively checks how each country is doing, producing “early warning” assessments to flag member states at risk of missing upcoming recycling goals [1]. Such oversight is essential to ensure that environmental directives are implemented effectively and that the EU’s collective goals (like reducing landfill use to under 10% by 2035 [1]) will be met.
 
This is where data-driven analysis becomes invaluable: by examining trends over time and patterns across countries, we help to reveal what’s working, what isn’t, and what factors might be driving these outcomes. With vast datasets collected (e.g. through Eurostat and national reports), we strive to make sense of the numbers – identifying successes to replicate, pinpointing gaps that need intervention, and ultimately supporting better policy decisions.
 
With this research project we want to answer the following questions:
- How has the amount of recycling of waste developed in the EU overall over time?
- How does recycling compare across countries in the EU?
- How does it compare for specific types of waste such as municipal waste, packaging waste, and
electrical waste?
- Are there characteristics of countries that could lead to increased recycling?
- How well can the development of the amount of recycling be predicted?
 
[1] [EUR-Lex COM(2023) 304](https://eur-lex.europa.eu/legal-content/EN/TXT/?%3Bqid=1686220362244&uri=COM%3A2023%3A304%3AFIN#:~:text=2025%2C%20Member%20States%20are%20required,waste%20management%20plans%20and%20waste)
 
[2] [Eurostat](https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Municipal_waste%20statistics#Municipal_waste_generation)
 
[3] [European Commission Environment](https://environment.ec.europa.eu/index_en?prefLang=de)

# Functions
This chapter contains all the functions that we created to help us analyze and clean the datasets.

In [ ]:
"""
This function filters a given Eurostat-dataframe to keep only the 27 current EU member states AND the EU27_2020 aggregate.
It then prints statistics and explicitly lists the countries/codes that were removed.
"""

def filter_to_eu27(df, name, eu_countries_map):
    """
    Filters the dataframe to keep only the 27 current EU member states AND the EU27_2020 aggregate.
    Prints statistics and explicitly lists the countries/codes that were removed.
    """
    initial_rows = len(df)
    
    # 1. Identify what is currently in the dataframe
    current_geo = set(df['geo'].unique())
    
    # 2. Identify valid keys (EU countries + EU Aggregate)
    valid_geo = set(eu_countries_map.keys())
    
    # 3. Calculate the difference (What will be removed?)
    removed_geo = current_geo - valid_geo
    
    # 4. Filter: Keep rows where 'geo' is in our valid keys
    df_filtered = df[df['geo'].isin(valid_geo)].copy()
    
    removed_rows = initial_rows - len(df_filtered)
    
    print(f"[{name}] Filtered to EU-27 + Aggregate:")
    print(f"   - Kept: {len(df_filtered)} rows")
    print(f"   - Removed: {removed_rows} rows")
    
    if removed_geo:
        # Sort list for better readability
        print(f"   - Removed Countries/Aggregates: {sorted(list(removed_geo))}")
    else:
        print(f"   - No countries removed (Dataset matched requirements exactly).")
    
    print("-" * 40) # Separator line
    
    return df_filtered

In [ ]:
"""
This function drops a given list of columns from a list of dataframes if the columns exist.
"""

def drop_columns_if_exist(dfs, columns_to_drop, verbose=True):
    """
    Drops a specific list of columns from a list of dataframes if they exist.
    """
    if verbose:
        print(f"--- Dropping {len(columns_to_drop)} specific columns ---")
    
    for i, df in enumerate(dfs):
        # Find intersection: which columns to drop are actually in this df?
        cols_in_df = [c for c in df.columns if c in columns_to_drop]
        
        if cols_in_df:
            df.drop(columns=cols_in_df, inplace=True)
            if verbose:
                print(f"[Dataset {i+1}] Dropped {len(cols_in_df)} columns: {cols_in_df}")
        else:
            if verbose:
                print(f"[Dataset {i+1}] No matching columns found.")

In [ ]:
"""
    This function removes all columns that are fully empty.
"""
def drop_fully_empty_columns(df, verbose=True):
    """
    Remove all columns that contain only NaN values.
    """
    empty_cols = df.columns[df.isna().all()].tolist()
    
    if verbose and empty_cols:
        print("Dropping fully empty columns:", empty_cols)
    elif verbose:
        print("No fully empty columns found.")
    
    return df.drop(columns=empty_cols)

In [ ]:
"""
This function identifies columns whic hexists in all dataframes and 
 """

def drop_global_constant_columns(dfs):
    """
    Identifies and drops columns that exist in ALL dataframes AND have the 
    exact same single constant value across ALL dataframes.
    Returns a dictionary of {column: constant_value} for documentation.
    """
    # 1. Identify columns present in ALL dataframes
    if not dfs:
        return {}
        
    common_cols = set(dfs[0].columns)
    for df in dfs[1:]:
        common_cols.intersection_update(df.columns)
    
    global_constants = {}
    
    # 2. Check which of these common columns are globally constant
    for col in list(common_cols):
        unique_vals = set()
        is_constant_everywhere = True
        
        for df in dfs:
            # Get unique values for this column in this dataframe
            vals = df[col].unique()
            
            # If a dataframe has more than 1 value (or 0), it's not constant within itself
            if len(vals) != 1:
                is_constant_everywhere = False
                break
            
            unique_vals.add(vals[0])
            
        # 3. Check if the value is the same across all dataframes (set length must be 1)
        if is_constant_everywhere and len(unique_vals) == 1:
            global_constants[col] = unique_vals.pop()

    # 4. Drop these columns from all dataframes
    if global_constants:
        print(f"Dropping {len(global_constants)} globally constant columns:")
        for col, val in global_constants.items():
            print(f"  - '{col}': '{val}'")
            for df in dfs:
                df.drop(columns=[col], inplace=True)
    else:
        print("No globally constant columns found.")
        
    return global_constants



In [ ]:
def extract_and_print_combined_legend(dfs):
    """
    Iterates through ALL dataframes, collects unique code-description pairs 
    across all of them, and prints a single consolidated legend.
    """
    # Define the known pairs (Code : Description Column)
    known_pairs = {
        'OBS_FLAG': 'Observation status (Flag) V2 structure',
        'unit': 'Unit of measure',
        'waste': 'Waste categories',
        'wst_oper': 'Waste management operations',
        'geo': 'Geopolitical entity (reporting)',
        'STRUCTURE_ID':	'STRUCTURE_NAME'
        
    }

    # Dictionary to store unique pairs for each category
    # Structure: {'OBS_FLAG': {('b', 'break in series'), ...}, ...}
    combined_legends = {key: set() for key in known_pairs}

    # 1. Collect all pairs from all dataframes
    for df in dfs:
        for code_col, desc_col in known_pairs.items():
            if code_col in df.columns and desc_col in df.columns:
                # Get unique pairs from this specific dataframe
                pairs = df[[code_col, desc_col]].dropna(subset=[code_col]).drop_duplicates()
                
                # Add to our master set
                for _, row in pairs.iterrows():
                    combined_legends[code_col].add((row[code_col], row[desc_col]))

    # 2. Print the consolidated legend
    print("=== COMBINED DATASET LEGEND (ALL DATASETS) ===\n")
    
    for category, pairs in combined_legends.items():
        if pairs:
            print(f"[{category}]")
            # Sort by code (first element of tuple) for readability
            sorted_pairs = sorted(list(pairs), key=lambda x: x[0])
            
            for code, desc in sorted_pairs:
                print(f"  {code} : {desc}")
            print() # Empty line between categories

In [ ]:
def create_wide_waste_dataframe_all_units(df):
    """
    Transforms the dataset to Wide Format retaining ALL units.
    Creates columns combining Operation + Unit (e.g., 'waste_generated_tonnes').
    Fixes the trailing underscore issue for index columns.
    """
    print("=== TRANSFORMING TO WIDE FORMAT (KEEPING ALL UNITS) ===\n")
    
    # 1. Pivot: Long -> Wide
    df_wide = df.pivot_table(
        index=['geo', 'TIME_PERIOD'], 
        columns=['wst_oper', 'unit'], 
        values='OBS_VALUE'
    ).reset_index()
    
    # 2. Flatten and Rename Columns
    
    op_map = {
        'GEN': 'waste_generated',
        'DSP_L_OTH': 'waste_landfill',
        'DSP_I': 'waste_incinerated',
        'RCY': 'waste_recycled_total',
        'TRT': 'waste_treated_total',
        'RCY_M': 'waste_recycled_material',
        'RCY_C_D': 'waste_composted',
        'PRP_REU': 'waste_reuse',
        'DSP_I_RCV_E': 'waste_incinerated_energy',
        'RCV_E': 'waste_energy_recovery'
    }
    
    unit_map = {
        'KG_HAB': 'kg_capita',
        'THS_T': 'tonnes'
    }
    
    new_columns = []
    
    for col in df_wide.columns:
        # Check if it is a tuple (MultiIndex column)
        if isinstance(col, tuple):
            op_code, unit_code = col
            
            # FIX: If unit_code is empty (happens for geo and TIME_PERIOD), just use the name
            if not unit_code:
                new_columns.append(op_code)
                continue
            
            # Translate codes
            op_name = op_map.get(op_code, op_code)
            unit_name = unit_map.get(unit_code, unit_code)
            
            # Combine only if unit exists
            new_name = f"{op_name}_{unit_name}"
            new_columns.append(new_name)
            
        else:
            # Fallback for simple strings
            new_columns.append(col)
    
    # Assign new clean column names
    df_wide.columns = new_columns
    
    # Info Output
    print(f"Pivoted shape: {df_wide.shape}")
    print(f"Columns created ({len(new_columns)}):")
    print(new_columns)
    
    return df_wide

In [ ]:
def create_waste_dataset_scaffold(df_ops, df_mun, df_pack, df_elect, country_map):
    """
    Create a balanced country–year panel (scaffold) and merge all datasets onto it.

    The function builds a complete grid of 28 countries × 24 years (2000–2023),
    then left‑joins municipal, packaging, e‑waste recycling rates and
    municipal waste operations onto this scaffold. This guarantees a
    perfectly balanced panel with 672 rows (one row per country–year).
    """
    print("=== CREATING MASTER DATASET (SCAFFOLD METHOD) ===\n")
    
    # --- A. CREATE SCAFFOLD (THE GRID) ---
    
    # 1. Define dimensions
    # Use the keys from the country_map to ensure the exact set of 28 entities
    geo_list = sorted(list(country_map.keys()))
    year_list = list(range(2000, 2024))  # 2000 to 2023
    
    expected_rows = len(geo_list) * len(year_list)
    print(f"Target dimensions: {len(geo_list)} entities * {len(year_list)} years = {expected_rows} rows")
    
    # 2. Build MultiIndex from cartesian product
    scaffold_index = pd.MultiIndex.from_product([geo_list, year_list], names=["geo", "TIME_PERIOD"])
    
    # 3. Create empty scaffold DataFrame
    df_scaffold = pd.DataFrame(index=scaffold_index).reset_index()
    
    # --- B. PREPARATION (RENAMING) ---
    
    # Municipal recycling rates
    mun_clean = df_mun[["geo", "TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"]].rename(columns={
        "OBS_VALUE": "recycling_rate_municipal_pc",
        "OBS_FLAG": "flag_municipal",
    })
    
    # Packaging recycling rates
    pack_clean = df_pack[["geo", "TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"]].rename(columns={
        "OBS_VALUE": "recycling_rate_packaging_pc",
        "OBS_FLAG": "flag_packaging",
    })
    
    # E‑waste recycling rates
    elect_clean = df_elect[["geo", "TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"]].rename(columns={
        "OBS_VALUE": "recycling_rate_ewaste_pc",
        "OBS_FLAG": "flag_ewaste",
    })
    
    # Waste operations (already wide). No additional filtering here; the join enforces the time window.
    ops_clean = df_ops.copy()
    
    # --- C. MERGING (LEFT JOINS ONTO THE SCAFFOLD) ---
    
    # 1. Scaffold + Municipal
    df_merged = pd.merge(df_scaffold, mun_clean, on=["geo", "TIME_PERIOD"], how="left")
    
    # 2. + Packaging
    df_merged = pd.merge(df_merged, pack_clean, on=["geo", "TIME_PERIOD"], how="left")
    
    # 3. + E‑Waste
    df_merged = pd.merge(df_merged, elect_clean, on=["geo", "TIME_PERIOD"], how="left")
    
    # 4. + Waste operations
    df_merged = pd.merge(df_merged, ops_clean, on=["geo", "TIME_PERIOD"], how="left")
    
    # --- D. FINAL STRUCTURE & REORDERING ---
    
    df_merged = df_merged.set_index(["geo", "TIME_PERIOD"]).sort_index()
    
    # Bring the key recycling indicators to the front
    cols_priority = [
        "recycling_rate_municipal_pc", "flag_municipal",
        "recycling_rate_packaging_pc", "flag_packaging",
        "recycling_rate_ewaste_pc", "flag_ewaste",
    ]
    cols_rest = [c for c in df_merged.columns if c not in cols_priority]
    df_merged = df_merged[cols_priority + cols_rest]
    
    print(f"Final shape: {df_merged.shape}")
    
    if len(df_merged) == expected_rows:
        print(f"SUCCESS: Dataset is perfectly balanced ({len(df_merged)} rows).")
    else:
        print(f"WARNING: Row mismatch! Expected {expected_rows}, got {len(df_merged)}.")
        
    return df_merged


## Read data

In [ ]:
# waste datasets
pack_waste = pd.read_csv("./data/ten00063_page_linear_2_0.csv", sep=",")
mun_waste = pd.read_csv("./data/sdg_11_60_linear_2_0.csv", sep=",")
elect_waste = pd.read_csv("./data/cei_wm060_linear_2_0.csv", sep=",")
waste_per_cap = pd.read_csv("./data/env_wasmun_linear_2_0.csv", sep=",")
batterie_waste = pd.read_csv("./data/LeadBatterieLife.csv", sep=";")
vehicle_waste = pd.read_csv("./data/vehicleEndOfLife.csv", sep=";")

# country characteristics
gdp_per_cap = pd.read_csv("./data/gdp-per-capita-worldbank.csv", sep=",")
urban_pop = pd.read_csv("./data/share-of-population-urban.csv", sep=",")
waste_expenditure = pd.read_csv("./data/IMF_COFOG_GENM_GF0501_WIDEF.csv", sep=",")
pop_density = pd.read_csv("./data/tps00003_linear_2_0.csv", sep=",")
educ = pd.read_csv("./data/edat_lfs_9903__custom_19291848_linear_2_0.csv", sep=",")

# extra datsets
join_year = pd.read_csv("./data/eu_joining_year.csv", sep=";")
waste_policies = pd.read_csv("./data/EU_waste_policies.csv", sep=";")

We created two datasets. One dataset contains only the waste datasets and looks into all variables in more detail. The second dataset contains all datasets, but there we only include the observed variable, the country and the year.

## Waste datasets
All waste datasets are from eurostat and have the same structure.

### Dataset Column Overview

| Column Name | Description |
| :--- | :--- |
| **STRUCTURE** | Dataflow definition reference (internal SDMX identifier). |
| **STRUCTURE_ID** | Technical ID of the Data Structure Definition (DSD) used (e.g., `ESTAT:TEN00063(1.0)`). |
| **STRUCTURE_NAME** | Human-readable name of the dataset structure. |
| **freq** | Code indicating the frequency of data collection (e.g., `A` for Annual). |
| **Time frequency** | Textual description of the frequency. |
| **waste** / **wst_oper** | Code identifying the specific waste category (e.g., `PLAS` for Plastic) or waste management operation (e.g., `GEN` for Generated). |
| **Waste categories** / **Waste operations** | Human-readable label for the waste category or operation code. |
| **unit** | Code for the unit of measurement (e.g., `PC` for %, `KG_HAB` for kg per capita). |
| **Unit of measure** | Textual description of the unit of measurement. |
| **geo** | Code representing the geopolitical entity (e.g., `AT`, `DE`, `EU27_2020`). |
| **Geopolitical entity (reporting)** | Full name of the country or region. |
| **TIME_PERIOD** | The reference period for the observation (e.g., `2022`). |
| **Time** | Textual representation of the reference period. |
| **OBS_VALUE** | The actual statistical value/measurement. |
| **Observation value** | Duplicate representation of the statistical value. |
| **OBS_FLAG** | Single-character code indicating data quality or status (e.g., `b` = break in time series, `e` = estimated). |
| **Observation status** | Textual description of the observation flag. |
| **CONF_STATUS** | Code indicating the confidentiality status of the observation. |
| **Confidentiality status** | Textual description of the confidentiality status. |

**Sources:**
*   [Eurostat Metadata > Code lists (Official)](https://ec.europa.eu/eurostat/web/metadata/code-lists)
*   [Eurostat SDMX InfoSpace (Standards)](https://ec.europa.eu/eurostat/web/sdmx-infospace/welcome)
*   [Eurostat API User Guide (Data Structure)](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-introduction)

In [ ]:
current_dfs = [pack_waste, mun_waste, elect_waste, waste_per_cap]
extract_and_print_combined_legend(current_dfs)

#### EU27_2020 Interpretation Across All Datasets

For all Eurostat datasets used in this project, the aggregate **`EU27_2020`** represents the value for the European Union as a whole in its current composition of 27 member states (i.e. all EU countries except the United Kingdom after Brexit). In each case, `EU27_2020` is **not** a simple arithmetic average of the 27 national values, but is derived from **aggregated quantities** and, where relevant, the **aggregated population**.

##### 1. Recycling rate of municipal waste (`sdg_11_60`)

For `sdg_11_60`, the code `EU27_2020` denotes the **EU‑wide recycling rate of municipal waste**. It is calculated by summing the recycled municipal waste of all 27 member states and dividing this by the sum of the total municipal waste generated in these countries, then multiplying by 100 to obtain a percentage. This yields a single EU‑level recycling rate that reflects the combined performance of all EU27_2020 countries.

Source: https://ec.europa.eu/eurostat/cache/metadata/en/sdg_11_60_esmsip2.htm

##### 2. Recycling rates for packaging waste (`ten00063`)

For the dataset `ten00063`, `EU27_2020` indicates the **EU‑wide recycling rate for packaging waste** for a given material (or for total packaging). The numerator is the sum of recycled packaging waste across all EU27_2020 countries, and the denominator is the sum of all generated packaging waste in those countries, again converted into a percentage. This ensures that larger countries contribute proportionally more to the EU aggregate than smaller ones.

Source: https://ec.europa.eu/eurostat/cache/metadata/en/env_waspac_esms.htm

##### 3. Recycling rate of WEEE (`cei_wm060`)

In the WEEE dataset `cei_wm060`, `EU27_2020` represents the **EU‑level recycling rate of separately collected electrical and electronic waste**. The EU indicator is computed as the sum of WEEE entering recycling and preparation‑for‑re‑use facilities across all member states, divided by the total amount of separately collected WEEE in the EU27_2020 countries. The resulting percentage captures the overall treatment efficiency of the EU’s WEEE management system rather than an average of national rates.

Source: https://ec.europa.eu/eurostat/cache/metadata/en/cei_wm060_esmsip2.htm

##### 4. Municipal waste by waste management operations (`env_wasmun`)

In the dataset `env_wasmun`, `EU27_2020` refers to **EU‑wide aggregates of municipal waste generation and treatment**. For absolute quantities (thousand tonnes), the EU value is obtained by summing the national amounts for all 27 member states. For per‑capita indicators (kg per inhabitant), the total EU waste amount is summed first and then divided by the aggregated EU27_2020 population for the respective year. This consistent aggregation principle applies to all operations (e.g. generated, landfilled, incinerated, recycled) and ensures that EU27_2020 reflects the combined waste flows of the entire Union.

Source: https://ec.europa.eu/eurostat/cache/metadata/en/env_wasmun_esms.htm


### Filter for Current EU Member States (EU27_2020)

For all core analyses, only the **current 27 EU member states (EU27_2020)** are included. This ensures consistency with the official Eurostat aggregate `EU27_2020` and avoids mixing in candidate, EFTA or former member states.

The following countries are retained in the datasets:

`['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark',
'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland',
'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands', 'Poland',
'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden']`

In [ ]:
eu_countries_map = {
    'EU27_2020': 'European Union - 27 countries (from 2020)',
    'AT': 'Austria',
    'BE': 'Belgium',
    'BG': 'Bulgaria',
    'HR': 'Croatia',
    'CY': 'Cyprus',
    'CZ': 'Czechia',
    'DK': 'Denmark',
    'EE': 'Estonia',
    'FI': 'Finland',
    'FR': 'France',
    'DE': 'Germany',
    'EL': 'Greece',
    'HU': 'Hungary',
    'IE': 'Ireland',
    'IT': 'Italy',
    'LV': 'Latvia',
    'LT': 'Lithuania',
    'LU': 'Luxembourg',
    'MT': 'Malta',
    'NL': 'Netherlands',
    'PL': 'Poland',
    'PT': 'Portugal',
    'RO': 'Romania',
    'SK': 'Slovakia',
    'SI': 'Slovenia',
    'ES': 'Spain',
    'SE': 'Sweden'
}




In [ ]:
# 1. Packaging Waste
pack_waste = filter_to_eu27(pack_waste, "Packaging Waste", eu_countries_map)

# 2. Municipal Waste
mun_waste = filter_to_eu27(mun_waste, "Municipal Waste", eu_countries_map)

# 3. E-Waste
elect_waste = filter_to_eu27(elect_waste, "E-Waste", eu_countries_map)

# 4. Waste Per Capita
waste_per_cap = filter_to_eu27(waste_per_cap, "Waste Per Capita", eu_countries_map)

### Drop fully empty columns

In [ ]:
pack_waste = drop_fully_empty_columns(pack_waste)
mun_waste = drop_fully_empty_columns(mun_waste)
elect_waste = drop_fully_empty_columns(elect_waste)
waste_per_cap = drop_fully_empty_columns(waste_per_cap)

### **Drop** **columns** which are globally contant over all datasets.

In [ ]:
all_dfs = [pack_waste, mun_waste, elect_waste, waste_per_cap]
dropped_info = drop_global_constant_columns(all_dfs)

Dropped 3 globally constant columns (these values are not any more relvant as they have across the datasets the same value):
  - **freq**: A
  - **Time frequency**: Annual
  - **STRUCTURE**: dataflow

### Drop columns with the same information

In [ ]:
# 1. List of Dataframes
all_dfs = [pack_waste, mun_waste, elect_waste, waste_per_cap]

# 2. List of columns strictly corresponding to the Legend created above
cols_to_remove_legend = [
    'Geopolitical entity (reporting)',          # Mapped to 'geo'
    'Unit of measure',                          # Mapped to 'unit'
    'Waste categories',                         # Mapped to 'waste'
    'Waste management operations',              # Mapped to 'wst_oper'
    'Observation status (Flag) V2 structure',    # Mapped to 'OBS_FLAG'
    'STRUCTURE_NAME'                           # Mapped to 'STRUCTURE_ID'
]

# 3. Execute Drop
drop_columns_if_exist(all_dfs, cols_to_remove_legend)

### Drop STRUCTURE_ID

**STRUCTURE_ID** (along with `STRUCTURE` and `STRUCTURE_NAME`) represents internal technical metadata used by Eurostat's database system (SDMX identifiers). These columns provide no additional analytical value, they are constant within each dataset and are removed to reduce redundancy and keep the dataframes clean.

In [ ]:
# 1. List of Dataframes
all_dfs = [pack_waste, mun_waste, elect_waste, waste_per_cap]

# 2. List of columns strictly corresponding to the Legend created above
cols_to_remove_legend = [
    'STRUCTURE_ID'
]

# 3. Execute Drop
drop_columns_if_exist(all_dfs, cols_to_remove_legend)

## Descriptive Statistics

In [ ]:
display(pack_waste)
print("\n--- DESCRIPTIVE STATISTICS ---")
print(pack_waste['OBS_VALUE'].describe())

display(mun_waste)
print("\n--- DESCRIPTIVE STATISTICS ---")
print(mun_waste['OBS_VALUE'].describe())

display(elect_waste)
print("\n--- DESCRIPTIVE STATISTICS ---")
print(elect_waste['OBS_VALUE'].describe())

display(waste_per_cap)
print("\n--- DESCRIPTIVE STATISTICS ---")
print(waste_per_cap['OBS_VALUE'].describe())

## Analyzing missing values

In [ ]:
print("\n--- MISSING VALUES ---")
print(pack_waste.isnull().sum())

print("\n--- MISSING VALUES ---")
print(mun_waste.isnull().sum())

print("\n--- MISSING VALUES ---")
print(elect_waste.isnull().sum())

print("\n--- MISSING VALUES (by Year) ---")
print(waste_per_cap.isnull().sum)

### Unit check

In [ ]:
print("\n--- UNITS CHECK ---")
print(pack_waste['unit'].value_counts())
print(pack_waste['waste'].value_counts())
pack_waste = pack_waste.drop(columns=['waste'])

print("\n--- UNITS CHECK ---")
print(mun_waste['unit'].value_counts())

print("\n--- UNITS CHECK ---")
print(elect_waste['unit'].value_counts())

print("\n--- OPERATION CATEGORIES CHECK ---")
print(waste_per_cap['wst_oper'].value_counts())
print("\n--- UNITS CHECK ---")
print(waste_per_cap['unit'].value_counts())
print("\n--- OBS_VALUE CHECK ---")
print(waste_per_cap['OBS_VALUE'].value_counts())

## Outlier check

In [ ]:
### Fehlt noch ###

### Data Transformation: Waste Operations (Pivot all Units)

We restructure the `waste_per_cap` dataset to the Wide Format, but **retain both units of measure** available in the raw data:
1.  **Kilograms per capita (`KG_HAB`)**: Essential for comparing countries relative to population size.
2.  **Thousand Tonnes (`THS_T`)**: Useful for analyzing the absolute environmental impact or mass.

**Transformation:**
We pivot the table using both `wst_oper` and `unit` as column headers. This creates distinct columns for every combination, e.g., `waste_generated_kg` and `waste_generated_tonnes`.

In [ ]:
waste_ops_wide = create_wide_waste_dataframe_all_units(waste_per_cap)

print("\n=== SAMPLE (WIDE FORMAT - ALL UNITS) ===")
display(waste_ops_wide)

### Explanation of the Single Columns

All indicators appear in two variants:
- `_kg_capita`: value expressed **per inhabitant** (kilograms per capita), suitable for cross-country comparison.
- `_tonnes`: value expressed as **absolute mass** (thousand tonnes), suitable for analysing total volumes over time.

#### 1. `waste_generated_kg_capita` / `waste_generated_tonnes`

These columns represent the **total amount of municipal waste generated** in a given country and year. Generation refers to waste handed over to the municipal collection system (or equivalent), mainly from households but also from commerce, offices and public institutions.

#### 2. `waste_treated_total_kg_capita` / `waste_treated_total_tonnes`

These indicators show the **total amount of municipal waste that underwent any treatment operation** (recovery or disposal) in that country and year. Treatment includes all operations after collection, such as recycling, composting, incineration and landfilling.

#### 3. `waste_landfill_kg_capita` / `waste_landfill_tonnes`

These columns capture municipal waste sent to **landfill or other final disposal operations** (codes D1–D7, D12 in EU waste legislation). Landfilling means depositing waste in or on land (e.g. landfill sites or surface impoundments) with no intention of recovery.

#### 4. `waste_incinerated_kg_capita` / `waste_incinerated_tonnes`

These indicators refer to municipal waste treated by **incineration as a disposal operation** (code D10), where the primary purpose is to destroy waste rather than to recover energy.

#### 5. `waste_incinerated_energy_kg_capita` / `waste_incinerated_energy_tonnes`

These indicators represent **incineration with energy recovery** (code R1), where waste is burned to generate useful energy (electricity, heat, or both) and the plant meets EU efficiency criteria.

#### 6. `waste_energy_recovery_kg_capita` / `waste_energy_recovery_tonnes`

These columns summarise **energy recovery from municipal waste**, typically corresponding to operations where waste replaces other fuels and the main outcome is usable energy.

#### 7. `waste_recycled_total_kg_capita` / `waste_recycled_total_tonnes`

These indicators show the **total amount of municipal waste recycled**, combining both material recycling and biological recycling (composting and anaerobic digestion). Recycling means reprocessing waste materials into products, materials or substances for the same or a different purpose, excluding energy recovery.

#### 8. `waste_recycled_material_kg_capita` / `waste_recycled_material_tonnes`

These columns quantify **material recycling**, where non-organic waste (e.g. paper, glass, metals, plastics) is reprocessed into new materials or products.

#### 9. `waste_composted_kg_capita` / `waste_composted_tonnes`

These indicators represent **biological recycling** of biodegradable municipal waste through **composting and anaerobic digestion**.

#### 10. `waste_reuse_kg_capita` / `waste_reuse_tonnes`

These columns measure **preparation for re-use**, i.e. operations such as checking, cleaning and repairing products or components so that they can be used again without further pre-processing.

---

**Sources:**

- Eurostat metadata “Municipal waste by waste management operations (env_wasmun_esms)”:  
  https://ec.europa.eu/eurostat/cache/metadata/en/env_wasmun_esms.htm

- Eurostat metadata “Management of waste by waste management operations and type of material (env_wassd_esms)”:  
  https://ec.europa.eu/eurostat/cache/metadata/en/env_wassd_esms.htm 

- Eurostat Statistics Explained – Municipal waste statistics:  
  https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Municipal_waste_statistics

- Eurostat product page env_wasmun:  
  https://ec.europa.eu/eurostat/product?code=env_wasmun&mode=view


## Merge of the waste datasets

# Data Aggregation (Master Dataset)

The four cleaned Eurostat datasets are consolidated into a single **master panel dataset** using a scaffold-based approach. This guarantees a balanced country–year structure and makes all indicators directly comparable across time and space.

A direct merge on the original datasets was not sufficient, because **none of the four sources provides complete coverage for all 28 entities and all years 2000–2023 simultaneously**. Each dataset has its own gaps in time and country coverage, so a simple intersection would either drop many years or exclude some countries. To avoid losing information and to keep the panel structurally consistent, a synthetic country–year scaffold is used as the backbone.

## Aggregation Strategy (Scaffold Method)

1. **Time window**  
   The analysis is restricted to the years **2000–2023**, in line with the defined project scope for the waste and recycling indicators.

2. **Scaffold construction (balanced panel)**  
   A complete **country–year grid** is created from:
   - 28 entities (the 27 EU member states plus the aggregate `EU27_2020`, or the selected country set for the analysis).
   - 24 years (2000–2023).  
   This results in a target of **28 × 24 = 672** rows. The scaffold DataFrame contains all combinations of `geo` (country code) and `TIME_PERIOD` (year), even if some indicators are missing for a given pair. Missing values in the final panel therefore represent genuine data gaps in the Eurostat series rather than structural holes in the table.

3. **Merging procedure**  
   The cleaned datasets are then merged onto this scaffold via sequential **left joins** on `geo` and `TIME_PERIOD`:
   - Municipal recycling rates (`sdg_11_60`),
   - Packaging recycling rates (`ten00063`),
   - E‑waste recycling rates (`cei_wm060`),
   - Municipal waste generation and treatment in wide format (`waste_ops_wide` / `env_wasmun`).  
   This ensures:
   - Every country–year combination in the scaffold appears exactly once.
   - All available indicators are attached where data exist, while the scaffold itself remains complete.

4. **Panel structure**  
   The final master dataset is indexed by a **multi-index** (`geo`, `TIME_PERIOD`), so that each row corresponds to one country–year combination and all waste‑ and recycling‑related variables are available as aligned columns.

## Pre-Merge Preparation: Renaming Strategy

To avoid column name collisions and keep the meaning of each variable explicit after merging, the following renaming scheme is applied to the recycling datasets before they are joined to the scaffold:

| Dataset            | Original Column | New Column Name                | Description                                              |
| :---------------- | :-------------- | :----------------------------- | :------------------------------------------------------- |
| **Packaging Waste** | `OBS_VALUE`    | `recycling_rate_packaging_pc`  | Packaging waste recycling rate (2025‑target definition, in percent). |
|                    | `OBS_FLAG`     | `flag_packaging`               | Eurostat quality/status flag for packaging data.        |
| **Municipal Waste** | `OBS_VALUE`    | `recycling_rate_municipal_pc`  | Recycling rate of municipal waste (SDG indicator, in percent). |
|                    | `OBS_FLAG`     | `flag_municipal`               | Eurostat quality/status flag for municipal data.        |
| **E‑Waste**         | `OBS_VALUE`    | `recycling_rate_ewaste_pc`     | Recycling rate of separately collected WEEE (in percent). |
|                    | `OBS_FLAG`     | `flag_ewaste`                  | 

This scaffold-based approach produces a **fully balanced master panel in terms of structure**, while transparently retaining the original data availability patterns from the underlying Eurostat sources.

In [ ]:
df_final_waste = create_waste_dataset_scaffold(waste_ops_wide, mun_waste, pack_waste, elect_waste, eu_countries_map)
display(df_final_waste)

# Now the second dataset

## Transform long datasets

In [ ]:
vehicle_waste_long = vehicle_waste.melt(
    id_vars=["Codes", "Labels"],      
    var_name="Year",                 
    value_name="Value"                
)
vehicle_waste_long["Year"] = vehicle_waste_long["Year"].astype(int)
vehicle_waste_long["Value"] = (
    vehicle_waste_long["Value"]
    .replace(":", pd.NA)        
    .str.replace(",", ".", regex=False)
)
vehicle_waste_long["Value"] = pd.to_numeric(vehicle_waste_long["Value"], errors="coerce")
vehicle_waste = vehicle_waste_long.sort_values(by=["Codes", "Year"]).reset_index(drop=True)

batterie_waste_long = batterie_waste.melt(
    id_vars=["Codes", "Labels"],      
    var_name="Year",                 
    value_name="Value"                
)
batterie_waste_long["Year"] = batterie_waste_long["Year"].astype(int)
batterie_waste_long["Value"] = (
    batterie_waste_long["Value"]
    .replace(":", pd.NA)        
    .str.replace(",", ".", regex=False)
)
batterie_waste_long["Value"] = pd.to_numeric(batterie_waste_long["Value"], errors="coerce")
batterie_waste = batterie_waste_long.sort_values(by=["Codes", "Year"]).reset_index(drop=True)


# Filter waste expenditure #
print(waste_expenditure.columns)
waste_expenditure = waste_expenditure[waste_expenditure["SECTOR"] == "IMF_SEC_GG"]
waste_expenditure = waste_expenditure[waste_expenditure["UNIT_MEASURE"] == "PT_GDP"]
cols_to_keep = (
    ['REF_AREA', 'REF_AREA_LABEL'] +
    [col for col in waste_expenditure.columns if col.isdigit()]
)
waste_expenditure = waste_expenditure[cols_to_keep]
print(waste_expenditure.columns)
waste_expenditure_long = waste_expenditure.melt(
    id_vars=["REF_AREA", "REF_AREA_LABEL"],     
    var_name="Year",                
    value_name="Value"                
)
waste_expenditure_long["Year"] = waste_expenditure_long["Year"].astype(int)
waste_expenditure_long["Value"] = pd.to_numeric(waste_expenditure_long["Value"], errors="coerce")
waste_expenditure = waste_expenditure_long.sort_values(by=["REF_AREA", "Year"]).reset_index(drop=True)

## Filter to only include EU countries

In [ ]:
eu_countries_map = {
    'EU27_2020': 'European Union - 27 countries (from 2020)',
    'AT': 'Austria',
    'BE': 'Belgium',
    'BG': 'Bulgaria',
    'HR': 'Croatia',
    'CY': 'Cyprus',
    'CZ': 'Czechia',
    'DK': 'Denmark',
    'EE': 'Estonia',
    'FI': 'Finland',
    'FR': 'France',
    'DE': 'Germany',
    'EL': 'Greece',
    'HU': 'Hungary',
    'IE': 'Ireland',
    'IT': 'Italy',
    'LV': 'Latvia',
    'LT': 'Lithuania',
    'LU': 'Luxembourg',
    'MT': 'Malta',
    'NL': 'Netherlands',
    'PL': 'Poland',
    'PT': 'Portugal',
    'RO': 'Romania',
    'SK': 'Slovakia',
    'SI': 'Slovenia',
    'ES': 'Spain',
    'SE': 'Sweden'
}

eu_countries_map_iso3 = {
    'EU27_2020': 'European Union - 27 countries (from 2020)',
    'AUT': 'Austria',
    'BEL': 'Belgium',
    'BGR': 'Bulgaria',
    'HRV': 'Croatia',
    'CYP': 'Cyprus',
    'CZE': 'Czechia',
    'DNK': 'Denmark',
    'EST': 'Estonia',
    'FIN': 'Finland',
    'FRA': 'France',
    'DEU': 'Germany',
    'GRC': 'Greece',
    'HUN': 'Hungary',
    'IRL': 'Ireland',
    'ITA': 'Italy',
    'LVA': 'Latvia',
    'LTU': 'Lithuania',
    'LUX': 'Luxembourg',
    'MLT': 'Malta',
    'NLD': 'Netherlands',
    'POL': 'Poland',
    'PRT': 'Portugal',
    'ROU': 'Romania',
    'SVK': 'Slovakia',
    'SVN': 'Slovenia',
    'ESP': 'Spain',
    'SWE': 'Sweden'
}


def filter_to_eu27(df, geo_col, name="Dataset", iso3 = False):
    """
    Filters the dataframe to keep only the 27 current EU member states AND the EU27_2020 aggregate.
    Prints statistics and explicitly lists the countries/codes that were removed.
    """
    if iso3:
        c_map = eu_countries_map_iso3
    else:
        c_map = eu_countries_map

    initial_rows = len(df)
    
    # 1. Identify what is currently in the dataframe
    current_geo = set(df[geo_col].dropna().unique())
    
    # 2. Identify valid keys (EU countries + EU Aggregate)
    valid_geo = set(c_map.keys())
    
    # 3. Calculate the difference (What will be removed?)
    removed_geo = current_geo - valid_geo
    
    # 4. Filter: Keep rows where geo_col is in our valid keys
    df_filtered = df[df[geo_col].isin(valid_geo)].copy()
    
    removed_rows = initial_rows - len(df_filtered)
    
    print(f"[{name}] Filtered to EU-27 + Aggregate:")
    print(f"   - Kept: {len(df_filtered)} rows")
    print(f"   - Removed: {removed_rows} rows")
    
    if removed_geo:
        
        print(f"   - Removed Countries/Aggregates: {sorted(list(removed_geo))}")
    else:
        print(f"   - No countries removed (Dataset matched requirements exactly).")
    
    print("-" * 40) 
    
    return df_filtered


In [ ]:
# Filter educ #
educ = educ[educ["isced11"] == "ED5-8"]

In [ ]:
# 1. Packaging Waste
pack_waste = filter_to_eu27(pack_waste, 'geo', "Packaging Waste")

# 2. Municipal Waste
mun_waste = filter_to_eu27(mun_waste, 'geo', "Municipal Waste")

# 3. E-Waste
elect_waste = filter_to_eu27(elect_waste, 'geo', "E-Waste")

# 4. Waste Per Capita
waste_per_cap = filter_to_eu27(waste_per_cap, 'geo', "Waste Per Capita")

# 5. Batterie waste
batterie_waste = filter_to_eu27(batterie_waste, 'Codes', "Batterie Waste")

# 6. Vecile waste
vehicle_waste = filter_to_eu27(vehicle_waste, 'Codes', "Vecile waste")

# 7. GDP per Capita
gdp_per_cap = filter_to_eu27(gdp_per_cap, 'Code', "GDP Per Capita", iso3 = True)

# 8. Urban population
urban_pop = filter_to_eu27(urban_pop, 'Code', "Urban population", iso3 = True)

# 9. Waste expenditure
waste_expenditure = filter_to_eu27(waste_expenditure, 'REF_AREA', "Waste expenditure", iso3 = True)

# 11. Population density
pop_density = filter_to_eu27(pop_density, 'geo', "Population density")

# 11. Education
educ = filter_to_eu27(educ, 'geo', "Education")

print("\nAll datasets restricted to EU-27 countries.")

## Join all datastes

In [ ]:
alpha3_to_alpha2 = {
    "AUT": "AT",
    "BEL": "BE",
    "BGR": "BG",
    "HRV": "HR",
    "CYP": "CY",
    "CZE": "CZ",
    "DNK": "DK",
    "EST": "EE",
    "FIN": "FI",
    "FRA": "FR",
    "DEU": "DE",
    "GRC": "EL",  # Greece is 'EL' in Eurostat
    "HUN": "HU",
    "IRL": "IE",
    "ITA": "IT",
    "LVA": "LV",
    "LTU": "LT",
    "LUX": "LU",
    "MLT": "MT",
    "NLD": "NL",
    "POL": "PL",
    "PRT": "PT",
    "ROU": "RO",
    "SVK": "SK",
    "SVN": "SI",
    "ESP": "ES",
    "SWE": "SE",
    "EU27_2020": "EU27_2020"  # keep EU aggregate as is
}

def to_alpha2(code):
    """Convert 3-letter country code to 2-letter for EU countries; otherwise keep as is."""
    return alpha3_to_alpha2.get(code, code)


pack_waste_fin = pack_waste[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(
    columns={"geo": "country","TIME_PERIOD": "year","OBS_VALUE": "value"}
)
mun_waste_fin = mun_waste[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(
    columns={"geo": "country","TIME_PERIOD": "year","OBS_VALUE": "value"}
)
elect_waste_fin = elect_waste[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(
    columns={"geo": "country","TIME_PERIOD": "year","OBS_VALUE": "value"}
)
waste_per_cap_fin = waste_per_cap[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(
    columns={"geo": "country","TIME_PERIOD": "year","OBS_VALUE": "value"}
)
batterie_waste_fin = batterie_waste[["Codes", "Year", "Value"]].rename(
    columns={"Codes": "country","Year": "year","Value": "value"}
)
vehicle_waste_fin = vehicle_waste[["Codes", "Year", "Value"]].rename(
    columns={"Codes": "country","Year": "year","Value": "value"}
)
waste_expenditure_fin = waste_expenditure[["REF_AREA", "Year", "Value"]].rename(
    columns={"REF_AREA": "country","Year": "year","Value": "value"}
)
gdp_per_cap_fin = gdp_per_cap[["Code", "Year", "GDP per capita, PPP (constant 2021 international $)"]].rename(
    columns={"Code": "country","Year": "year","GDP per capita, PPP (constant 2021 international $)": "value"}
)
urban_pop_fin = urban_pop[["Code", "Year", "Urban population (% of total population)"]].rename(
    columns={"Code": "country","Year": "year","Urban population (% of total population)": "value"}
)
pop_density_fin = pop_density[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(
    columns={"geo": "country","TIME_PERIOD": "year","OBS_VALUE": "value"}
)
educ_fin = educ[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(
    columns={"geo": "country","TIME_PERIOD": "year","OBS_VALUE": "value"}
)
join_year = join_year.rename(
    columns={"geo": "country","joining_year": "joining_year"}
)

display(educ_fin)


In [ ]:
from functools import reduce
pack_waste_fin      = pack_waste_fin.rename(columns={"value": "packaging_recycling"})
mun_waste_fin       = mun_waste_fin.rename(columns={"value": "municipal_recycling"})
elect_waste_fin     = elect_waste_fin.rename(columns={"value": "electrical_recycling"})
waste_per_cap_fin   = waste_per_cap_fin.rename(columns={"value": "waste_per_capita"})
batterie_waste_fin  = batterie_waste_fin.rename(columns={"value": "battery_recycling"})
vehicle_waste_fin   = vehicle_waste_fin.rename(columns={"value": "vehicle_recycling"})
waste_expenditure_fin   = waste_expenditure_fin.rename(columns={"value": "waste_expenditure"})
gdp_per_cap_fin     = gdp_per_cap_fin.rename(columns={"value": "gdp_per_capita"})
urban_pop_fin       = urban_pop_fin.rename(columns={"value": "urban_population_share"})
pop_density_fin     = pop_density_fin.rename(columns={"value": "population_density"})
educ_fin            = educ_fin.rename(columns={"value": "education_attainment"})
dfs = [pack_waste_fin, mun_waste_fin, elect_waste_fin,
      batterie_waste_fin, vehicle_waste_fin, gdp_per_cap_fin, urban_pop_fin, pop_density_fin, educ_fin, waste_expenditure_fin]  
#i = 1
for df in dfs:
    df["country"] = df["country"].apply(to_alpha2)
    #print(i)
    #print(df["country"].nunique())
    #i = i+1

df_merged = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=["country", "year"],
        how="outer"
    ),
    dfs
)
df_merged = df_merged[
    (df_merged["year"] >= 2000) &
    (df_merged["year"] <= 2023)
].sort_values(["country", "year"]).reset_index(drop=True)

df_merged = df_merged.merge(
    join_year,           
    on="country",           
    how="left"              
)
df_merged["joining_year"] = pd.to_numeric(df_merged["joining_year"], errors="coerce")
df_merged["joining_year"] = df_merged["joining_year"].astype(float).astype('Int64')
# Source - https://stackoverflow.com/a
# Posted by Abhishek Bhatia
# Retrieved 2025-12-29, License - CC BY-SA 4.0

df_merged.to_csv("./data/merged_dataset.csv", index=False)

In [ ]:
duplicates = (
    df_merged
    .groupby(["country", "year"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
)

if duplicates.empty:
    print("✅ Jede (country, year)-Kombination kommt genau einmal vor.")
else:
    print("❌ Es gibt doppelte (country, year)-Kombinationen:")
    display(duplicates)


# Number of countries #
print(df_merged["country"].nunique())


In [ ]:
#print(waste_policies.columns)
df_merged_all = df_merged.copy()

for abbr in waste_policies["abbreviation"].unique():
    df_merged_all[abbr] = 0
#df_merged_all


In [ ]:

for abbr in waste_policies["abbreviation"].unique(): 
    df_merged_all[abbr] = 0 
    for abbr, pol_group in waste_policies.groupby("abbreviation"): 
        for _, pol in pol_group.iterrows(): 
            year_adopted = pol["year_adopted"] 
            if pol["eu_wide"] == "yes": 
                mask = df_merged_all["year"] >= year_adopted 
            else: 
                mask = ( (df_merged_all["country"] == pol["country"]) & (df_merged_all["year"] >= year_adopted) ) 
            df_merged_all.loc[mask, abbr] = 1

In [ ]:
df_merged_all.to_csv("./data/merged_dataset_all.csv", index=False)

## Analyzing missing values 


In [ ]:
missing_by_country = (
    df_merged
    .groupby("country")
    .apply(lambda x: x.isna().sum())
)
missing_by_country = missing_by_country.drop(["country", "year", "joining_year"], axis = 1)
missing_by_country

plt.figure(figsize=(8, 6))
sns.heatmap(
    missing_by_country,
    cmap="Reds",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Missing proportion"}
)

plt.title("Proportion of Missing Values by Country and Variable")
plt.xlabel("Variable")
plt.ylabel("Country")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
df_long = df_merged.melt(
    id_vars=["country", "year"],
    var_name="variable",
    value_name="value"
)
missing_by_year = (
    df_long
    .groupby("year")["value"]
    .apply(lambda x: x.isna().mean())
    .reset_index(name="missing")
)
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

sns.lineplot(
    data=missing_by_year,
    x="year",
    y="missing",
    marker="o"
)

plt.title("Proportion of Missing Values Over Time")
plt.xlabel("Year")
plt.ylabel("Missing proportion")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


missing_var_year = (
    df_long
    .groupby(["variable", "year"])["value"]
    .apply(lambda x: x.isna().mean())
    .reset_index(name="missing")
)
heatmap_var_year = missing_var_year.pivot(
    index="variable",
    columns="year",
    values="missing"
)
plt.figure(figsize=(14, 8))

sns.heatmap(
    heatmap_var_year,
    cmap="Reds",
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"label": "Missing proportion"}
)

plt.title("Proportion of Missing Values by Variable and Year")
plt.xlabel("Year")
plt.ylabel("Variable")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()




In [ ]:
missing_country_year = (
    df_long
    .groupby(["country", "year"])["value"]
    .apply(lambda x: x.isna().mean())
    .reset_index(name="missing")
)

heatmap_country_year = missing_country_year.pivot(
    index="country",
    columns="year",
    values="missing"
)

plt.figure(figsize=(14, 8))

sns.heatmap(
    heatmap_country_year,
    cmap="Reds",
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"label": "Missing proportion"}
)

plt.title("Proportion of Missing Values by Country and Year")
plt.xlabel("Year")
plt.ylabel("Country")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
df_merged['waste_expenditure'] = df_merged['waste_expenditure']*100
df_merged_all['waste_expenditure'] = df_merged_all['waste_expenditure']*100

In [ ]:
df_merged

## Analyzing outliers

In [ ]:
numeric_vars = (
    df_merged
    .select_dtypes(include=[np.number])
    .drop(["year", "joining_year"], axis = 1)
)
numeric_vars

df_long = df_merged.melt(
    value_vars=numeric_vars,
    var_name="variable",
    value_name="value"
)

g = sns.catplot(
    data=df_long,
    x="variable",
    y="value",
    kind="box",
    col="variable",
    col_wrap=4,
    sharey=False,
    height=3
)

g.set_titles("{col_name}")
g.set_axis_labels("", "Value")

# REMOVE x-axis
for ax in g.axes.flat:
    ax.set_xticks([])
    ax.set_xlabel("")

# g = sns.catplot(
#     data=df_long,
#     x="variable",
#     y="value",
#     kind="box",
#     col="variable",
#     col_wrap=4,
#     sharey=False,
#     height=3
# )

# g.set_titles("{col_name}")
# g.set_axis_labels("", "Value")

In [ ]:
selected_vars = [
    "packaging_recycling", "municipal_recycling", "electrical_recycling", "battery_recycling", "vehicle_recycling", 
    "urban_population_share", "education_attainment", "waste_expenditure"
]
df_year = (
    df_merged
    .groupby("year")[selected_vars]
    .mean()
    .reset_index()
)
df_year_long = df_year.melt(
    id_vars="year",
    var_name="variable",
    value_name="value"
)
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.lineplot(
    data=df_year_long,
    x="year",
    y="value",
    hue="variable",
    marker="o"
)

plt.title("Trends Over Time (Selected Variables)")
plt.xlabel("Year")
plt.ylabel("Value")

plt.legend(title="Variable")
plt.tight_layout()
plt.show()



In [ ]:
df_year[selected_vars] = (
    df_year[selected_vars]
    .rolling(window=3, center=True)
    .mean()
)
df_year_long = df_year.melt(
    id_vars="year",
    var_name="variable",
    value_name="value"
)
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.lineplot(
    data=df_year_long,
    x="year",
    y="value",
    hue="variable",
    marker="o"
)

plt.title("Trends Over Time (Selected Variables)")
plt.xlabel("Year")
plt.ylabel("Value")

plt.legend(title="Variable")
plt.tight_layout()
plt.show()


In [ ]:
df_year_long = df_year.melt(
    id_vars="year",        # the column to keep fixed
    value_vars=selected_vars,  # columns to melt
    var_name="variable",
    value_name="value"
)



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.lineplot(
    data=df_year_long,
    x="year",
    y="value",
    hue="variable",
    marker="o"
)

plt.title("Trends Over Time (Selected Variables)")
plt.xlabel("Year")
plt.ylabel("Value")
plt.legend(title="Variable")
plt.tight_layout()
plt.show()


In [ ]:
rate_indicators = ["packaging_recycling", "municipal_recycling", "electrical_recycling", "battery_recycling", "vehicle_recycling"]
#rate_indicators = [
#    "packaging_recycling", "municipal_recycling", "electrical_recycling", "battery_recycling", "vehicle_recycling", 
#    "urban_population_share", "education_attainment", "waste_expenditure"
#]

rates_long = df_merged.melt(
    id_vars=[],                 # no id columns in your R example
    value_vars=rate_indicators, # columns to melt
    var_name="indicator",
    value_name="value"
)
rates_long["indicator"] = rates_long["indicator"].astype(str)

plt.figure(figsize=(10, 6))

sns.histplot(
    data=rates_long,
    x="value",
    hue="indicator",
    multiple="dodge",
    binwidth=5,
    edgecolor="black",
    legend=True  # ensure legend is requested
)

# Vertical line at 100
plt.axvline(x=100, color="red", linewidth=1, linestyle="solid")

plt.title("Distribution of Enrollment Rates")
plt.xlabel("Rate (%)")
plt.ylabel("Count")


plt.tight_layout()
plt.show()

#colors = ["green", "red", "violet", "skyblue", "orange"]


## Correlation-Analysis

To gain an initial overview of the relationships between variables, pooled Pearson correlations are computed using all available country–year observations. These correlations reflect average associations across EU countries and years, without accounting for country-specific or time-specific effects.

In addition, pairwise correlations between the predictor variables used in later analysis are examined to assess potential multicollinearity and to better understand the relationships among the explanatory variables. For this purpose, correlations are calculated separately within each country using over-time variation and are then summarized across countries. High correlations may indicate redundant information among predictors.
Absolute correlation coefficients are used to assess the strength of associations between predictors, as the objective is to identify potential multicollinearity. Averaging signed correlations could mask strong but oppositely signed relationships across countries.

##  Pooled Pearson Correlations

In [ ]:

corr = df_merged[selected_vars].corr()
print(corr)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()




The correlation analysis shows the following:
- Packaging and municipal recycling are strongly correlated (r ≈ 0.72), suggesting that countries that perform well in one type tend to perform well in the other.

- Packaging and municipal recycling are also positively correlated with higher GDP per capita, higher education attainment, and higher urban population share.
    - For example, packaging recycling correlates with GDP per capita (r ≈ 0.36), education (r ≈ 0.43), and urban population share (r ≈ 0.24).
    - Municipal recycling shows even stronger correlations with GDP (r ≈ 0.60), education (r ≈ 0.47), and urban population (r ≈ 0.36).

- Population density shows a negative correlation with packaging (r ≈ -0.31) and electrical recycling (r ≈ -0.21), suggesting that denser countries tend to recycle less in these categories.

- Waste expenditure does not show strong correlations with any of the recycling types.

- Furthermore, explanatory variables are themselves correlated. GDP per capita, education attainment, and urban population share are positively correlated with each other.


## Mean Predictor Correlations

In [ ]:

selected_predictors = ['gdp_per_capita', 'urban_population_share', 'population_density',  'waste_expenditure', 'education_attainment']  

# Threshold for "high correlation"
CORR_THRESHOLD = 0.80

country_correlations = {}
# Extract unique countries from the MultiIndex (first level)
for country in panel_df.index.get_level_values(0).unique():
    country_data = panel_df.loc[country, selected_predictors].dropna() 
    if len(country_data) > 1:  # Need at least 2 observations for corr.
        country_correlations[country] = country_data.corr()

# Extract all unique pairs
all_pairs = {}
for i in range(len(selected_predictors)):
    for j in range(i+1, len(selected_predictors)):
        var1 = selected_predictors[i]
        var2 = selected_predictors[j]
        pair_key = (var1, var2)
        all_pairs[pair_key] = []

# Collect correlation values for each pair across all countries
for country, corr_mat in country_correlations.items():
    for i in range(len(selected_predictors)):
        for j in range(i+1, len(selected_predictors)):
            var1 = selected_predictors[i]
            var2 = selected_predictors[j]
            pair_key = (var1, var2)
            r = abs(corr_mat.loc[var1, var2])
            all_pairs[pair_key].append(r)


print("\nPairwise Correlation Summary (across all countries):\n")
print(f"{'Variable 1':<25} {'Variable 2':<25} {'Mean |r|':<10} {'High in % of countries':<10}")

high_corr_pairs = []

for (var1, var2), corr_values in sorted(all_pairs.items(), key=lambda x: np.mean(x[1]), reverse=True):
    mean_r = np.mean(corr_values)
    pct_high = 100 * sum(1 for r in corr_values if r > CORR_THRESHOLD) / len(corr_values)
    
    print(f"{var1:<25} {var2:<25} {mean_r:<10.2f} {pct_high:10.2f}%")


# Draw mean coorrelation heatmap
mean_corr_matrix = pd.DataFrame(0.0, index=selected_predictors, columns=selected_predictors)
for (var1, var2), corr_values in all_pairs.items():
    mean_r = np.mean(corr_values)
    mean_corr_matrix.loc[var1, var2] = mean_r
    mean_corr_matrix.loc[var2, var1] = mean_r

# Fill diagonal
for v in selected_predictors:
    mean_corr_matrix.loc[v, v] = 1.0

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(mean_corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Mean Predictor Correlations (Across All Countries)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

The pairwise correlation analysis across EU countries shows that many socio-economic predictors are strongly related within countries over time. In particular, the following variables are highly correlated:
- Urban population share and education attainment (|r| = 0.92, high in 85% of countries)
- GDP per capita and urban population (|r| = 0.83) 
- GDP per capita and education attainment (|r| = 0.83)
- Population density and education attainment (|r| = 0.81)
- Population density and urban population share (|r| = 0.80)
- Population density and GDP per capita (|r| = 0.79)

In contrast, waste expenditure shows relatively weak correlations with the other predictors. 
These results indicate that many of the country-level characteristics are closely related.

Why these correlations are higher than pooled correlations:
These correlations were calculated within each country over time and then averaged across countries using absolute values. This preserves the strength of associations even if the direction of correlation differs across countries, and avoids cancellation of opposing trends. 

# Recycling Development

## How has the amount of recycling of waste developed in the EU overall over time?
 
 
To answer this question comprehensively, we analyze the **EU27_2020 aggregate** (representing the 27-member European Union in its current composition) along two key dimensions:
 
1. **Efficiency (Recycling Rates in %)**: How effectively is waste being recycled? We examine three waste streams:
   - **Municipal Waste**: Household and similar commercial waste
   - **Packaging Waste**: Materials like paper, glass, metal, and plastic packaging
   - **E-Waste**: Electronic equipment waste (treatment efficiency of collected devices)
 
2. **Absolute Volume (Mass in Tonnes)**: What is the actual composition of waste treatment? We track:
   - **Recycling** (material recovery + composting) – most desirable
   - **Energy Recovery** (incineration with heat/electricity generation) – recovery operation
   - **Incineration** (without energy recovery) – disposal
   - **Landfill** – least desirable disposal method
   - **Waste Treatment** the overall treatment of waste
   - **Waste Generation** the overall generation of waste
 
 
**Methodological Note on Data Measurement:** For packaging waste, Eurostat implemented a stricter measurement methodology starting from reference year 2020. This change results in lower but more accurate recycling rates.
- Packaging waste by waste management operations (env_waspac):  
  https://ec.europa.eu/eurostat/cache/metadata/en/env_waspac_esms.htm
 
 
 
 
 
Packaging waste by waste management operations (env_waspac)
 

In [ ]:
# 1. Extract data (if not already done)
# We use .copy() to avoid warnings
df_final = df_final_waste.copy()
df_final_waste = df_final_waste.loc['EU27_2020'].copy()

# 2. Setup
 
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 7))
 
 
# 1. Municipal Waste
sns.lineplot(ax=ax, data=df_final_waste, x=df_final_waste.index, y='recycling_rate_municipal_pc',
             label='Municipal Waste (Ref: Generated)', color='#1f77b4', linewidth=3, marker='o')
 
# 2. Packaging Waste
sns.lineplot(ax=ax, data=df_final_waste, x=df_final_waste.index, y='recycling_rate_packaging_pc',
             label='Packaging Waste (Ref: Generated)', color='#ff7f0e', linewidth=2.5, linestyle='--', marker='s')
 
# 3. E-Waste
sns.lineplot(ax=ax, data=df_final_waste, x=df_final_waste.index, y='recycling_rate_ewaste_pc',
             label='E-Waste (Treatment Efficiency)*', color='#2ca02c', linewidth=2.5, linestyle='-.', marker='^')
 
 
ax.set_title('Development of Recycling Rates in the EU (2000-2023)', fontsize=16, fontweight='bold', pad=20)
ax.set_ylabel('Recycling Rate (%)', fontsize=12)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylim(0, 105) # Etwas Platz oben für die Textbox
ax.grid(True, linestyle='--', alpha=0.7)
 
# X-Axis: Show every year
ax.set_xticks(df_final_waste.index)
ax.set_xticklabels(df_final_waste.index, rotation=45)
 
# Box for explanation of the E-Waste rate
textstr = (
    r'$\bf{Context\ for\ E-Waste:}$' + '\n'
    'Rate refers to treatment efficiency of\n'
    r'$\it{collected}$ devices only.'
)
props = dict(boxstyle='round', facecolor='#eaffea', alpha=0.9, edgecolor='#2ca02c')
ax.text(0.02, 0.96, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)
 
# Legend positioning
ax.legend(loc='lower right', fontsize=11, frameon=True)
 
plt.tight_layout()
plt.show()

In [ ]:
#df_eu27 = df_final.loc['EU27_2020'].copy()

# 1. Setup Plot
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(14, 8))
 
# 2. Prepare Data
years = df_final_waste.index
rec = df_final_waste['waste_recycled_total_tonnes'].fillna(0)
energy = df_final_waste['waste_energy_recovery_tonnes'].fillna(0)
incineration = df_final_waste['waste_incinerated_tonnes'].fillna(0)
landfill = df_final_waste['waste_landfill_tonnes'].fillna(0)
total_treated = df_final_waste['waste_treated_total_tonnes']
total_generated = df_final_waste['waste_generated_tonnes']
 
# 3. Plotting: Stacked Bars
p1 = ax.bar(years, rec, label='Recycling (Material & Compost)', color='#2ca02c', alpha=0.9, edgecolor='white')
p2 = ax.bar(years, energy, bottom=rec, label='Energy Recovery', color='#ff7f0e', alpha=0.9, edgecolor='white')
p3 = ax.bar(years, incineration, bottom=rec+energy, label='Incineration (Disposal)', color='#d62728', alpha=0.9, edgecolor='white')
p4 = ax.bar(years, landfill, bottom=rec+energy+incineration, label='Landfill (Disposal)', color='#7f7f7f', alpha=0.9, edgecolor='white')
 
# 4. Plotting: Reference Lines
ax.plot(years, total_treated, color='black', linestyle='--', linewidth=1.5, marker='o', markersize=4,
        label='Total Treated (Sum of Bars)')
ax.plot(years, total_generated, color='purple', linestyle='-', linewidth=3, marker='o', markersize=5,
        label='Total Generated (Waste Arising)')
 
# 5. Formatting
ax.set_title('Composition of Municipal Waste in the EU (2000-2023)', fontsize=16, fontweight='bold', pad=20)
ax.set_ylabel('Mass (Tonnes)', fontsize=12)
ax.set_xlabel('Year', fontsize=12)
ax.legend(loc='upper left', fontsize=10, frameon=True, bbox_to_anchor=(1, 1))
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45)
 
plt.tight_layout()
plt.show()
 
import pandas as pd
 
# 1) Set index and clean
if 'TIME_PERIOD' in df_final_waste.columns:
    df_final_waste.set_index('TIME_PERIOD', inplace=True)
df_final_waste.index = pd.to_numeric(df_final_waste.index, errors='coerce')
df_final_waste.sort_index(inplace=True)
 
# 2) Define tonnes columns
cols_mass = [
    'waste_recycled_total_tonnes',
    'waste_energy_recovery_tonnes',
    'waste_incinerated_tonnes',
    'waste_landfill_tonnes'
]
 
# 3) Define total tonnes
total_col = 'waste_treated_total_tonnes'
 
# 4) Create result table
df_table = pd.DataFrame(index=df_final_waste.index)
 
# 5) Calculate shares (%)
labels = ['Recycling (%)', 'Energy Recovery (%)', 'Incineration (%)', 'Landfill (%)']
 
for col, label in zip(cols_mass, labels):
    df_table[label] = (df_final_waste[col] / df_final_waste[total_col]) * 100
 
# 6) Round values
df_table = df_table.round(1)
 
# 7) Add total tonnes
df_table['Total Treated (Tonnes)'] = df_final_waste[total_col].round(0).astype('Int64', errors='ignore')
 
# 8) Show table
print("Detailed table: treatment shares (%)")
display(df_table)
df_table.to_csv('waste_treatment_shares_table.csv')
 
 
 
 

In [ ]:
def plot_mean_std(metric_col: str, title: str):
    """
    Plot yearly mean ±1 std across EU countries (excl. EU27_2020) for the given metric column.
    """
    df_plot = df_final.reset_index().query("geo != 'EU27_2020'").copy()
    df_plot["TIME_PERIOD"] = df_plot["TIME_PERIOD"].astype("Int64")

    stats = (
        df_plot.groupby("TIME_PERIOD")[metric_col]
        .agg(["mean", "std"])
        .reset_index()
    )
    stats['EU27_2020'] = df_final.reset_index().query("geo == 'EU27_2020'")[['TIME_PERIOD', metric_col]].set_index('TIME_PERIOD')[metric_col].values


    plt.figure(figsize=(10, 5))
    sns.lineplot(data=stats, x="TIME_PERIOD", y="mean", marker="o", color="blue", label="Mean")
    sns.lineplot(data=stats, x="TIME_PERIOD", y="EU27_2020", marker="o", color="red", label="EU27_2020")
    plt.fill_between(
        stats["TIME_PERIOD"],
        stats["mean"] - stats["std"],
        stats["mean"] + stats["std"],
        color="blue",
        alpha=0.2,
        label="±1 std"
    )
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.title(title)
    plt.xlabel("Year")
    plt.ylabel("Recycling Rate (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_mean_std("recycling_rate_municipal_pc", "Municipal Recycling Rate (Mean ±1 SD)")
plot_mean_std("recycling_rate_packaging_pc", "Packaging Recycling Rate (Mean ±1 SD)")
plot_mean_std("recycling_rate_ewaste_pc", "E-Waste Recycling Rate (Mean ±1 SD)")

Conclusion fehlt noch

## How does recycling compare across countries in the EU?
We now examine country-level trends to compare how recycling rates have evolved across EU member states.

In [ ]:
def plot_each_country(type_of_waste, type_of_waste_label):
    sns.set(style="whitegrid")
    df_plot = df_final.reset_index().copy()
    df_plot["TIME_PERIOD"] = df_plot["TIME_PERIOD"].astype("Int64")  # avoid floats like 2012.5
    plt.figure(figsize=(14, 6))
    sns.lineplot(
        data=df_plot,
        x="TIME_PERIOD",
        y="recycling_rate_" + type_of_waste + "_pc",
        hue="geo",
        marker="o",
    )
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.title(type_of_waste_label + " Recycling Rates Over Time by Country")
    plt.xlabel("Year")
    plt.ylim(bottom=-2)
    plt.ylabel("Recycling Rate (%)")
    plt.legend(title="Country", bbox_to_anchor=(0.5, -0.15), loc="upper center", ncol=6)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_each_country('ewaste', 'E-Waste Recycling')
plot_each_country('municipal', 'Municipal Recycling')
plot_each_country('packaging', 'Packaging Recycling')

| Metric | Typical Range | Trend | Stability |
| :--- | :--- | :--- | :--- |
| **Municipal** | 10% – 70% | Consistent Growth | High |
| **Packaging** | 40% – 85% | Stagnant / Fluctuating | Moderate |
| **E-Waste** | 60% – 100%+ | High but Volatile | Low (Data Outliers) |
 
There is one country showing an E-Waste recycling rate of 120%. This is due to the hoarding effect.
 
Let's have a look at the most interesting countries e.g. those, that show the highest improvements and those which have shown declining recycling rates.

In [ ]:
def plot_top_bottom_countries(type_of_waste, type_of_waste_label, top_n=3):
    sns.set(style="whitegrid")
    df_plot = df_final.reset_index().copy()
    df_plot["TIME_PERIOD"] = df_plot["TIME_PERIOD"].astype("Int64")

    # Calculate improvement using each country's first and last available data points
    improvements = {}
    improvement_details = []
    
    for country in df_plot['geo'].unique():
        data_country = df_plot[df_plot['geo'] == country]
        
        # Get data for this country and waste type, sorted by year
        country_data = data_country[['TIME_PERIOD', 'recycling_rate_' + type_of_waste + '_pc']].dropna()
        
        if len(country_data) >= 2:  # Need at least 2 data points
            country_data = country_data.sort_values('TIME_PERIOD')
            
            start_year = country_data.iloc[0]['TIME_PERIOD']
            start_value = country_data.iloc[0]['recycling_rate_' + type_of_waste + '_pc']
            
            end_year = country_data.iloc[-1]['TIME_PERIOD']
            end_value = country_data.iloc[-1]['recycling_rate_' + type_of_waste + '_pc']
            
            improvement = end_value - start_value
            improvements[country] = improvement
            
            # Store details for summary table
            improvement_details.append({
                'Country': eu_countries_map.get(country, country),
                'Code': country,
                'Start Year': int(start_year),
                'Start Rate (%)': round(start_value, 2),
                'End Year': int(end_year),
                'End Rate (%)': round(end_value, 2),
                'Absolute Change (pp)': round(improvement, 2),
                'Relative Change (%)': round((improvement / start_value) * 100, 2) if start_value > 0 else None
            })

    # Sort countries by improvement
    sorted_countries = sorted(improvements.items(), key=lambda x: x[1], reverse=True)
    top_countries = [country for country, _ in sorted_countries[:top_n]]
    bottom_countries = [country for country, _ in sorted_countries[-top_n:]]

    selected_countries = top_countries + bottom_countries

    # Create summary table for selected countries
    summary_df = pd.DataFrame(improvement_details)
    summary_df = summary_df[summary_df['Code'].isin(selected_countries)]
    
    # Sort to show top performers first, then bottom performers
    summary_df['Order'] = summary_df['Code'].map({code: i for i, code in enumerate(top_countries + bottom_countries)})
    summary_df = summary_df.sort_values('Order').drop('Order', axis=1)
    
    # Add category column
    summary_df['Category'] = summary_df['Code'].apply(
        lambda x: 'Top Performer' if x in top_countries else 'Bottom Performer'
    )
    summary_df = summary_df[['Category', 'Country', 'Code', 'Start Year', 'Start Rate (%)', 
                             'End Year', 'End Rate (%)', 'Absolute Change (pp)', 'Relative Change (%)']]

    # Plot
    plt.figure(figsize=(14, 6))
    sns.lineplot(
        data=df_plot[df_plot['geo'].isin(selected_countries)],
        x="TIME_PERIOD",
        y="recycling_rate_" + type_of_waste + "_pc",
        hue="geo",
        marker="o",
    )
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.title(f"Top {top_n} and Bottom {top_n} Countries in {type_of_waste_label} Recycling Rates Over Time")
    plt.xlabel("Year")
    plt.ylim(bottom=-2)
    plt.ylabel("Recycling Rate (%)")
    plt.legend(title="Country", bbox_to_anchor=(0.5, -0.15), loc="upper center", ncol=6)
    plt.tight_layout()
    plt.show()
    
    # Display summary table
    print(f"\n=== PERFORMANCE SUMMARY: {type_of_waste_label} Recycling ===\n")
    display(summary_df.reset_index(drop=True))
    
    return None

In [ ]:
plot_top_bottom_countries('ewaste', 'E-Waste Recycling')
plot_top_bottom_countries('municipal', 'Municipal Recycling')
plot_top_bottom_countries('packaging', 'Packaging Recycling')

### Analysis
While a deep dive into every individual regulation is outside our current scope, we can examine the "big picture" laws that correlate with the most significant shifts in waste recycling rates across the top and bottom performer nations. The goal is to find out which policies work best to propose rolling them out in other countries as well. Further, we want to explain the bottom performers problems as well to prevent mistakes in other countries.
 
#### Municipal Waste
| Country       | Policy/External Factor                            | Implementation Year    | Effect                                                                                                                                                                                                                            | Source (URL)                                                                                                                                                                                                             |
| :-----------: | :------------------------------------------------ | :--------------------: | :-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | :----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Austria(*)**   | Landfill Ordinance (*Deponieverordnung*)          | 1989, Full Ban: 2004         | Banned the disposal of untreated municipal waste, which forced a shift to thermal/biological treatment and led to the aggressive rollout of separate bio-waste collection systems.                                                | [Landfill Tax, Incineration Tax and Landfill Ban in Austriai](https://ieep.eu/wp-content/uploads/2022/12/AT-Landfill-Tax-final.pdf)                                                                                      |
| **Slovenia**  | Door-to-Door Collection & Pay-As-You-Throw (PAYT) | 2014 (Ljubljana Model) | Inverted the convenience hierarchy and introduced volume-based pricing for residual waste, driving behavioral change and resulting in a 95% decrease in residual waste sent to landfill in the capital.                           | [Zero-waste strategy: Ljubljana, Slovenia](https://www.ebrdgreencities.com/policy-tool/zero-waste-strategy-ljubljana-slovenia/)                                                                                          |
| **Lithuania** | Deposit Return System (DRS)                       | 2016                   | A technology-enabled system with a uniform €0.10 deposit and "return-to-retail" mandate, which delivered a profound and immediate increase in plastic bottle collection rates from 34% to over 90%.                               | [Race towards Deposit Return Systems](https://fairresourcefoundation.org/en/race-towards-deposit-return-systems/)                                                                                                        |
| **Latvia**    | Natural Resources Tax (NRT) Escalator             | Started around 2017    | Legislated a steep, multi-year increase in the landfill tax (up to €110/tonne by 2024), which served as "shock therapy" to price landfilling out of the market and drive material diversion.                                      | [Latvia - European Environment Agency](https://www.eea.europa.eu/en/topics/in-depth/waste-and-recycling/municipal-and-packaging-waste-management-country-profiles-2025/lv-municipal-waste-factsheet.pdf/@@download/file) |
| **Sweden**    | Landfill Bans on Combustible & Organic Waste      | 2002 & 2005            | Effectively eliminated the landfilling of municipal solid waste, leading to the rapid construction of Waste-to-Energy plants and creating an "incineration trap" that presents a structural barrier to higher material recycling. | [Municipal waste management in Sweden](https://www.eea.europa.eu/publications/managing-municipal-solid-waste/sweden-municipal-waste-management)                                                                          |
| **Bulgaria**  | Statistical Pivot (EU Decision 2019/1004)         | Effective Post-2020    | A change in EU calculation methodology requiring reporting of *net* waste entering final recycling, which caused a statistical "drop" that exposed lower-than-claimed actual material recycling capacity.                         | [Bulgaria - European Environment Agency](https://www.eea.europa.eu/publications/many-eu-member-states/bulgaria)                                                                                                          |
 
(*) While Austria maintains high recycling rates (over 60%), its plateauing performance, particularly in plastic recycling (The Plastic Gap), stems from a reliance on incineration similar to Sweden's "Incineration Trap," a challenge it is adressing now via the National Circular Economy Strategy in December 2022, aiming to increase the circularity rate of the economy from 9.7% to 18% by 2030.
[Impact story: from the CGR Austria to the first National Circular Economy Strategy](https://www.circularity-gap.world/updates-collection/impact-story-from-the-cgr-austria-to-the-first-national-circular-economy-strategy)
 
#### Packaging Waste
Note, a methodological purification mandated by the EU in 2019 forced countries to measure recycling at the final reprocessing input, thereby removing statistical buffers and leading to a structural decline or stagnation in reported rates.
[Commission Implementing Decision (EU) 2019/665](https://eur-lex.europa.eu/eli/dec_impl/2019/665/oj/eng)
 
| Country     | Policy/External Factor                                                            | Implementation Year | Effect                                                                                                                                                                                   | Source (URL)                                                                                                                                                                                                             |
| :---------- | :-------------------------------------------------------------------------------- | :------------------ | :--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | :----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Poland**  | "Garbage Revolution" (Act on Maintaining Cleanliness and Order in Municipalities) | 2013                | Break in series; massive jump in reported collection due to nationalization of waste ownership, bringing previously 'invisible' waste into the formal system.                            | [EEA Factsheet](https://www.eea.europa.eu/en/topics/in-depth/waste-and-recycling/country-profiles-on-waste-prevention-2025/poland_waste-prevention-factsheet-2024/@@download/file)                                       |
| **Romania** | DNA Investigation into OIREP Fraud                                                | 2016                | Catastrophic drop (correction) in reported rates as widespread fraud within the Producer Responsibility Organization (OIREP) system was exposed and fake data was removed.               | [Eurostat - Packaging waste statistics](https://ec.europa.eu/eurostat/statistics-explained/index.php/Packaging_waste_statistics)                                                                                         |
| **Romania** | RetuRO Deposit Return System                                                      | 2023                | Expected sharp increase in beverage container recycling due to the implementation of a new, transparent, and digitally traceable closed-loop collection system.                          | [The Guardian](https://www.theguardian.com/environment/2025/nov/27/we-like-it-a-lot-how-romania-created-the-largest-deposit-return-scheme-in-the-world)                                                                  |
| **Ireland** | EPA Reclassification of Cement Kiln Fuel                                          | \~2012-14           | Decrease/Stagnation in reported rates as the use of waste fuel in cement kilns was reclassified from "recycling" to "energy recovery," removing it from the recycling statistics.        | [Ireland - European Environment Agency](https://www.eea.europa.eu/publications/many-eu-member-states/ireland)                                                                                                            |
| **Greece**  | EOAN Audits & "Blue Bin" Correction                                               | 2017-19             | Decline/Stagnation as stricter regulatory audits enforced the deduction of high impurity/residue rates (often 40-50%) from the total Blue Bin collected tonnage.                         | [Greece - European Environment Agency](https://www.eea.europa.eu/en/topics/in-depth/waste-and-recycling/municipal-and-packaging-waste-management-country-profiles-2025/lv-municipal-waste-factsheet.pdf/@@download/file) |
| **Latvia**  | Natural Resources Tax (NRT) & Exemptions                                          | Ongoing             | High sustained rates due to a punitive tax on non-compliant producers, creating a powerful "tax-or-recycle" financial motive to ensure high collection and recycling via PROs.           | [Latvia - European Environment Agency](https://www.eea.europa.eu/en/topics/in-depth/waste-and-recycling/municipal-and-packaging-waste-management-country-profiles-2025/lv-municipal-waste-factsheet.pdf/@@download/file)                                                                                                              |
| **Cyprus**  | Stockpiling & Export timing (Statistical Anomaly)                                 | 2022                | Artificial spike in steel recycling due to a one-time export of accumulated metal scrap, which was recorded in that year's statistics, decoupling the rate from current-year generation. | [Cyprus - European Environment Agency](https://www.eea.europa.eu/en/topics/in-depth/waste-and-recycling/municipal-and-packaging-waste-management-country-profiles-2025/lv-municipal-waste-factsheet.pdf/@@download/file)                                                                                                              |
 
 
 
#### E-Waste
| Country     | Policy/External Factor                      | Implementation Year | Effect (in essence)                                                                                                                               | Source (URL)                                                                                                                                                                                    |
| :---------: | :-----------------------------------------: | :-----------------: | :-----------------------------------------------------------------------------------------------------------------------------------------------: | :---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------: |
| **Sweden**  | Phased Landfill Ban                         | 2002                | Forced the rapid development of rigorous upstream separation streams for WEEE and fueled Waste-to-Energy networks.                                | [Waste incineration in the Nordic countries](https://pub.norden.org/temanord2024-524/3-synthesis-of-results.html)                                                                               |
| **Denmark** | Statutory Clearing House Model (DPA-System) | Post-2002           | Mathematically allocates WEEE collection responsibility among schemes, preventing "cherry-picking" and ensuring geographic equity.                | [WEEE system setup](https://www.diva-portal.org/smash/get/diva2:1552417/FULLTEXT01.pdf)                                                                                                         |
| **France**  | Visible Fee (Eco-participation)             | 2005                | Secured a ring-fenced revenue stream for Producer Responsibility Organizations (PROs) and drove immediate public awareness of recycling costs.    | [WEEE Compliance](https://fr.commscope.com/corporate-responsibility-and-sustainability/environment/weee-recycling-in-france/)                                                                   |
| **France**  | "1-for-0" Take-Back Obligations             | 2014                | Mandated large retailers to accept small WEEE (\<25 cm) without a new purchase, drastically increasing the density of the collection network.     | [Assessment of WEEE collection systems and their effectiveness in other European countries](https://dts.valpak.co.uk/Content/Documents/Eunomia%20Valpak%20DTS%20Report%202019.pdf)              |
| **Spain**   | Royal Decree 110/2015                       | 2015                | Mandated a centralized electronic platform for data traceability to combat the loss of WEEE to the informal "grey market."                        | [In-depth review of the WEEE Collection Rates and Targets - SCYCLE](https://www.scycle.info/wp-content/uploads/2020/11/In-depth-review_WEEE-Collection-Targets-and-Rates_UNITAR_2020_Final.pdf) |
| **Greece**  | 2008-2014 Financial Crisis                  | 2010                | Caused a "Denominator Collapse" (precipitous drop in new sales/POM), which statistically distorted collection rates and led to consumer hoarding. | [Update of WEEE Collection Rates, Targets, Flows, and Hoarding – 2021](https://weee-forum.org/wp-content/uploads/2022/12/Update-of-WEEE-Collection_web_final_nov_29.pdf)                        |
| **Malta**   | Civic Amenity Sites Infrastructure Rollout  | 2007                | Acted as a "release valve" for decades of hoarded historical WEEE, causing collection volumes and reported rates to occasionally exceed 100%.     | [Update of WEEE Collection Rates, Targets, Flows, and Hoarding – 2021](https://weee-forum.org/wp-content/uploads/2022/12/Update-of-WEEE-Collection_web_final_nov_29.pdf)                        |
 
#### Conclusion
Those countries that show stagnation are usually on a very high level and face certain limitations of their previously implemented policies. Most of the declines can be explained by a methodological shift or external factor (E.g. rapid change of E-Waste recycling in Malta can be explained by the hoarding effect) - careful analysis required. Also a huge part of municipal waste is packaging waste which is the reason why those policies are very similar.
 
What seems to have worked best for **municipal waste** is the
- landfill tax (see Austria).
- residual waste tax (see Slovenia).
- deposit return system (see Lithuania).
 
However, note that in combination they are most effective (e.g. landfill tax promotes incineration (results in incineration trap e.g. Austria, Sweden); to fight incineration trap you could introduce a deposit return system.)
 
For **packaging waste** (highly correlated with municipal waste):
- give ownership to the municipalities (this applies not only to the packaing waste but Poland has shown great performance of recycling increase in packaging waste due to this policy).
- deposit system (see Romania, Latvia)
 
 

### Target Gap Analysis
Building on the EU targets outlined earlier, we now examine how individual member countries are performing against these benchmarks.

In [ ]:
def plot_latest_recycling_bars(df_final):
    """
    Side-by-side bar chart of the last available recycling % per country
    for municipal and packaging waste, with EU target lines.
    Returns a small summary DataFrame of the used values/years.
    """
    df = df_final.reset_index()

    # EU27 countries (ISO codes), exclude EU27_2020 aggregate
    eu_iso = sorted([code for code in eu_countries_map.keys() if code != 'EU27_2020'])

    df = df[df['geo'].isin(eu_iso)]

    def last_metric(metric):
        tmp = (df[['geo','TIME_PERIOD',metric]]
               .dropna(subset=[metric])
               .sort_values(['geo','TIME_PERIOD']))
        last = tmp.groupby('geo').last(numeric_only=False).reset_index()
        last.rename(columns={metric: f'{metric}_val', 'TIME_PERIOD': f'{metric}_year'}, inplace=True)
        return last

    m_last = last_metric('recycling_rate_municipal_pc')
    p_last = last_metric('recycling_rate_packaging_pc')

    latest = pd.merge(m_last, p_last, on='geo', how='outer')

    # Sort by municipal values (fallback to packaging if municipal missing)
    sort_key = latest['recycling_rate_municipal_pc_val'].fillna(latest['recycling_rate_packaging_pc_val'])
    latest = latest.loc[sort_key.sort_values(ascending=False).index]

    # Plot
    sns.set(style='whitegrid')
    fig, ax = plt.subplots(figsize=(14, 6))
    x = range(len(latest))
    width = 0.42

    ax.bar([i - width/2 for i in x],
           latest['recycling_rate_municipal_pc_val'],
           width=width, color='#1f77b4', label='Municipal (last %)')

    ax.bar([i + width/2 for i in x],
           latest['recycling_rate_packaging_pc_val'],
           width=width, color='#ff7f0e', label='Packaging (last %)')

    # X labels with last years per metric
    labels = []
    for _, row in latest.iterrows():
        m_year = row.get('recycling_rate_municipal_pc_year')
        p_year = row.get('recycling_rate_packaging_pc_year')
        if pd.notna(m_year) and pd.notna(p_year):
            labels.append(f"{row['geo']} (M:{int(m_year)}, P:{int(p_year)})")
        elif pd.notna(m_year):
            labels.append(f"{row['geo']} (M:{int(m_year)})")
        elif pd.notna(p_year):
            labels.append(f"{row['geo']} (P:{int(p_year)})")
        else:
            labels.append(row['geo'])
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=45, ha='right')

    # EU targets (from your description)
    ax.axhline(50, color='blue', linestyle='--', linewidth=1, label='Municipal target 50% (2020)')
    ax.axhline(60, color='blue', linestyle='-.', linewidth=1, label='Municipal target 60% (2030)')
    ax.axhline(65, color='blue', linestyle=':', linewidth=1, label='Municipal target 65% (2035)')
    ax.axhline(65, color='orange', linestyle='--', linewidth=1.5, label='Packaging target 65% (2025)')

    ax.set_ylim(0, 100)
    ax.set_ylabel('Recycling rate (%)')
    ax.set_title('Latest available recycling rates by country (Municipal vs Packaging)')
    ax.legend(ncol=2, bbox_to_anchor=(1.0, 1.0))
    plt.tight_layout()
    plt.show()

    return latest[['geo',
                   'recycling_rate_municipal_pc_val','recycling_rate_municipal_pc_year',
                   'recycling_rate_packaging_pc_val','recycling_rate_packaging_pc_year']].rename(columns={
        'recycling_rate_municipal_pc_val':'municipal_pc',
        'recycling_rate_municipal_pc_year':'municipal_year',
        'recycling_rate_packaging_pc_val':'packaging_pc',
        'recycling_rate_packaging_pc_year':'packaging_year'
    })

# Usage
latest_summary = plot_latest_recycling_bars(df_final)
display(latest_summary)

#  Are there characteristics of countries that could lead to increased recycling?
Three methods are used to explore possible answers to this question: a regression model that reflects the datas panel structure, PCA and Hierarchical Clustering.
For this, the previously established merged_dataset for the EU 27 countries is used. 

In [ ]:
# drop rows with missing keys and set MultiIndex
panel_df = df_merged.dropna(subset=['country', 'year']).copy()
panel_df = panel_df.loc[panel_df['country'] != 'EU27_2020']  # Exclude EU aggregate
panel_df['year'] = panel_df['year'].astype(int)
panel_df = panel_df.set_index(['country', 'year']).sort_index()
panel_df.head(3)

In [ ]:
selected_outcomes = ['packaging_recycling', 'municipal_recycling', 'electrical_recycling']  
selected_predictors = ['gdp_per_capita', 'urban_population_share', 'population_density', 'education_attainment', 'waste_expenditure'] 

## Panel Regression
To examine whether country characteristics are associated with increased recycling, a two-way fixed-effects panel regression model is employed. The dataset consists of multiple EU countries observed over several years. The panel regression allows to investigate the following: When a country’s characteristics change, does its recycling rate change, when compared to that same country in other years?

The analysis focuses on within-country changes over time, estimating how changes in country characteristics are associated with changes in recycling outcomes, while controlling for unobserved, time-invariant country-specific factors. These factors are captured through country fixed effects.

In addition, year fixed effects are included to account for common shocks and trends affecting all countries in a given year, such as EU-wide policy changes.

Separate regressions are estimated for each recycling outcome. All predictors are standardized to allow for comparability across variables. Standard errors are clustered at the country level.

Observations with missing values in either the dependent variable or any of the predictor variables are excluded prior to estimating the panel regression models, since panel regression requires complete observations.

### Multivariate Panel Regression with different Lags
We investigate whether GDP, education attainment, and waste expenditure have delayed rather than immediate effects on recycling rates. Therefore, we test 1-year and 3-year lagged versions of these variables and compare them to current (non-lagged) values.

In [ ]:
# Create lagged versions of each variable
panel_df_lags = panel_df.copy()
panel_df_lags['gdp_lag1'] = panel_df_lags.groupby(level=0)['gdp_per_capita'].shift(1)
panel_df_lags['gdp_lag3'] = panel_df_lags.groupby(level=0)['gdp_per_capita'].shift(3)
panel_df_lags['educ_lag1'] = panel_df_lags.groupby(level=0)['education_attainment'].shift(1)
panel_df_lags['educ_lag3'] = panel_df_lags.groupby(level=0)['education_attainment'].shift(3)
panel_df_lags['waste_lag1'] = panel_df_lags.groupby(level=0)['waste_expenditure'].shift(1)
panel_df_lags['waste_lag3'] = panel_df_lags.groupby(level=0)['waste_expenditure'].shift(3)

# Run regressions for current and lagged versions

results_current = {}
results_gdp_lag1 = {}
results_gdp_lag3 = {}
results_educ_lag1 = {}
results_educ_lag3 = {}
results_waste_lag1 = {}
results_waste_lag3 = {}

for outcome in selected_outcomes:
    # Current values
    y = panel_df[outcome]
    X = panel_df[selected_predictors].copy()
    valid = (~y.isna()) & (~X.isna().any(axis=1))
    X_scaled = pd.DataFrame(StandardScaler().fit_transform(X[valid]), index=y[valid].index, columns=X.columns)
    results_current[outcome] = PanelOLS(y[valid], X_scaled, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
    
    # GDP lag1
    X_temp = panel_df_lags[['gdp_lag1', 'urban_population_share', 'population_density', 'education_attainment', 'waste_expenditure']].copy()
    valid = (~y.isna()) & (~X_temp.isna().any(axis=1))
    X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_temp[valid]), index=y[valid].index, columns=X_temp.columns)
    results_gdp_lag1[outcome] = PanelOLS(y[valid], X_scaled, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
    
    # GDP lag3
    X_temp = panel_df_lags[['gdp_lag3', 'urban_population_share', 'population_density', 'education_attainment', 'waste_expenditure']].copy()
    valid = (~y.isna()) & (~X_temp.isna().any(axis=1))
    X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_temp[valid]), index=y[valid].index, columns=X_temp.columns)
    results_gdp_lag3[outcome] = PanelOLS(y[valid], X_scaled, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
    
    # Education lag1
    X_temp = panel_df_lags[['gdp_per_capita', 'urban_population_share', 'population_density', 'educ_lag1', 'waste_expenditure']].copy()
    valid = (~y.isna()) & (~X_temp.isna().any(axis=1))
    X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_temp[valid]), index=y[valid].index, columns=X_temp.columns)
    results_educ_lag1[outcome] = PanelOLS(y[valid], X_scaled, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
    
    # Education lag3
    X_temp = panel_df_lags[['gdp_per_capita', 'urban_population_share', 'population_density', 'educ_lag3', 'waste_expenditure']].copy()
    valid = (~y.isna()) & (~X_temp.isna().any(axis=1))
    X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_temp[valid]), index=y[valid].index, columns=X_temp.columns)
    results_educ_lag3[outcome] = PanelOLS(y[valid], X_scaled, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
    
    # Waste lag1
    X_temp = panel_df_lags[['gdp_per_capita', 'urban_population_share', 'population_density', 'education_attainment', 'waste_lag1']].copy()
    valid = (~y.isna()) & (~X_temp.isna().any(axis=1))
    X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_temp[valid]), index=y[valid].index, columns=X_temp.columns)
    results_waste_lag1[outcome] = PanelOLS(y[valid], X_scaled, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
    
    # Waste lag3
    X_temp = panel_df_lags[['gdp_per_capita', 'urban_population_share', 'population_density', 'education_attainment', 'waste_lag3']].copy()
    valid = (~y.isna()) & (~X_temp.isna().any(axis=1))
    X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_temp[valid]), index=y[valid].index, columns=X_temp.columns)
    results_waste_lag3[outcome] = PanelOLS(y[valid], X_scaled, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)

# Compare t-statistics

print("All lag specifications (t-statistics):\n")

for outcome in selected_outcomes:
    print(f"{outcome}:")
    
    # GDP
    t_curr = results_current[outcome].tstats.get('gdp_per_capita', 0)
    t_lag1 = results_gdp_lag1[outcome].tstats.get('gdp_lag1', 0)
    t_lag3 = results_gdp_lag3[outcome].tstats.get('gdp_lag3', 0)
    print(f"  GDP:     current t={t_curr:6.2f}, lag1 t={t_lag1:6.2f}, lag3 t={t_lag3:6.2f}")
    
    # Education Attainment
    t_curr = results_current[outcome].tstats.get('education_attainment', 0)
    t_lag1 = results_educ_lag1[outcome].tstats.get('educ_lag1', 0)
    t_lag3 = results_educ_lag3[outcome].tstats.get('educ_lag3', 0)
    print(f"  Educ:    current t={t_curr:6.2f}, lag1 t={t_lag1:6.2f}, lag3 t={t_lag3:6.2f}")
    
    # Waste Expenditure
    t_curr = results_current[outcome].tstats.get('waste_expenditure', 0)
    t_lag1 = results_waste_lag1[outcome].tstats.get('waste_lag1', 0)
    t_lag3 = results_waste_lag3[outcome].tstats.get('waste_lag3', 0)
    print(f"  Waste:   current t={t_curr:6.2f}, lag1 t={t_lag1:6.2f}, lag3 t={t_lag3:6.2f}\n")


- Packaging recycling:
  - GDP: all strongly significant; no lag effect
  - Education: current is best
  - Waste: all |t| < 1.96 (not significant)

- Municipal recycling:
  - GDP/education/waste: all |t| < 1.96; lags don’t help

- Electrical recycling:
  - GDP: all |t| < 1.96; no lag effect
  - Education: all |t| < 1.96; no lag effect
  - Waste: lag1 has largest |t| but still weak



Since current (non-lagged) values are as strong or stronger for almost all cases, we decided not to use lagged variables in the final model. This suggests that the effects of GDP, education, and waste expenditure on recycling rates operate with minimal time delays, or that any delayed effects are absorbed by the country and year fixed effects in our panel regression model.

Note: Statistical significance at 5% (|t| >= 1.96)

In [ ]:
results = {}

# Scale predictors 
scaler = StandardScaler()

for outcome in selected_outcomes:
    y = panel_df[outcome]
    X = panel_df[selected_predictors].copy()

    # Drop rows with missing outcome or predictors
    valid_idx = (~y.isna()) & (~X.isna().any(axis=1))
    y2 = y.loc[valid_idx]
    X2 = X.loc[valid_idx]

    # Scale predictors
    X2_scaled = pd.DataFrame(
        scaler.fit_transform(X2.values),
        index=X2.index,
        columns=X2.columns,
    )
    
    mod = PanelOLS(y2, X2_scaled, entity_effects=True, time_effects=True)
    res = mod.fit(cov_type='clustered', cluster_entity=True)
    results[outcome] = res
    print(f"\n=== {outcome} ===")
    print(res.summary)

# Summarize which predictors are significant
print("="*80)
print("Significant Predictors by Waste Type (p < 0.05)")
print("="*80)

for outcome in selected_outcomes:
    if outcome in results:
        res = results[outcome]
        print(f"\n{outcome}:")
        
        sig_found = False
        for predictor in selected_predictors:
            if predictor in res.pvalues.index:
                pval = res.pvalues[predictor]
                if pval < 0.05:
                    coef = res.params[predictor]
                    tstat = res.tstats[predictor]
                    print(f"  {predictor:<30} coef={coef:>7.3f}, t={tstat:>6.2f}, p={pval:.4f}")
                    sig_found = True


Since urban population share never has a significant effect on the different recycling types, and it has high correlations with gdp_per_capita and  population_density, we drop it from the model.

### Reduced Model:
To handle missing values in the dataset, we investigate two strategies:
- Deletion of missing values
- Imputation of missing values

For imputation, the following approach is applied:
- Linear interpolation for missing values in the middle of a country’s time series
- Forward and backward filling for small gaps at the beginning or end of a series
- Replacement of any remaining missing values with the within-country mean

We compare results from both approaches to assess the influence of the explanatory variables on recycling rates.

In [ ]:

reduced_predictors = ['gdp_per_capita', 'population_density', 'education_attainment', 'waste_expenditure']  

results_deletion = {}
scaler_reduced = StandardScaler()

for outcome in selected_outcomes:
    y = panel_df[outcome]
    X = panel_df[reduced_predictors].copy()

    valid_idx = (~y.isna()) & (~X.isna().any(axis=1))
    y2 = y.loc[valid_idx]
    X2 = X.loc[valid_idx]

    X2_scaled = pd.DataFrame(
        scaler_reduced.fit_transform(X2.values),
        index=X2.index,
        columns=X2.columns,
    )
    
    mod = PanelOLS(y2, X2_scaled, entity_effects=True, time_effects=True)
    res_del = mod.fit(cov_type='clustered', cluster_entity=True)
    results_deletion[outcome] = res_del
    #print(f"\n=== {outcome} (deletion) ===")
    #print(res_del.summary)

In [ ]:


df_cy = panel_df.copy()

## Imputate missing values in training set ##
# Interpolations if some variables are missing in the middle of the time series
df_cy[reduced_predictors] = df_cy.groupby("country")[reduced_predictors].transform(
    lambda x: x.interpolate(method="linear", limit_area="inside")
)
#Forward and backwardsfill, if we have some small gaps at the beginning and the end
df_cy[reduced_predictors] = df_cy.groupby("country")[reduced_predictors].ffill(limit=3)
df_cy[reduced_predictors] = df_cy.groupby("country")[reduced_predictors].bfill(limit=3)

#Impute remaining missing values with within-country mean
for col in reduced_predictors:
    df_cy[col + "_missing"] = df_cy[col].isna().astype(int)
    country_means = df_cy.groupby("country")[col].transform("mean")
    df_cy[col] = df_cy[col].fillna(country_means)

results_imputation = {}

for outcome in selected_outcomes:
    y = df_cy[outcome]
    X = df_cy[reduced_predictors].copy()

    valid_idx = ~y.isna()
    y2 = y.loc[valid_idx]
    X2 = X.loc[valid_idx]

    X2_scaled = pd.DataFrame(
        scaler_reduced.fit_transform(X2.values),
        index=X2.index,
        columns=X2.columns,
    )
    
    mod = PanelOLS(y2, X2_scaled, entity_effects=True, time_effects=True)
    res_imp = mod.fit(cov_type='clustered', cluster_entity=True)
    results_imputation[outcome] = res_imp
    #print(f"\n=== {outcome} (imputation) ===")
    #print(res_imp.summary)




In [ ]:
waste_types = selected_outcomes  
rows = []

for waste in waste_types:
    # Deletion
    res_del = results_deletion[waste]
    del_df = pd.DataFrame({
        'Variable': res_del.params.index,
        'b_del': res_del.params.values,
        'StdErr_del': res_del.std_errors.values,
        't_del': res_del.tstats.values,
        'p_del': res_del.pvalues.values
    })

    # Imputation
    res_imp = results_imputation[waste]
    imp_df = pd.DataFrame({
        'Variable': res_imp.params.index,
        'b_imp': res_imp.params.values,
        'StdErr_imp': res_imp.std_errors.values,
        't_imp': res_imp.tstats.values,
        'p_imp': res_imp.pvalues.values
    })

    merged = pd.merge(del_df, imp_df, on='Variable')
    merged.insert(0, 'Waste type', waste)

    rows.append(merged)

comparison_table = pd.concat(rows, ignore_index=True)

comparison_table[comparison_table.select_dtypes(include='number').columns] = comparison_table.select_dtypes(include='number').round(3)

display(comparison_table)


1. Packaging Recycling:

- Deletion results:

    - gdp_per_capita: significant negative effect (b = -4.92, t = -3.28, p = 0.001) → higher GDP associated with lower packaging recycling
    - population_density: significant negative effect (b = -5.83, t = -2.19, p = 0.03) → denser populations slightly reduce packaging recycling
    - education_attainment and waste_expenditure: not significant (p > 0.4)

- Imputation results:
    - None of the predictors are significant (all p > 0.1)
    - Coefficients are smaller in magnitude 

Imputation increased sample size and smoothed missing data, which reduced the apparent significance of GDP and population density for packaging recycling.


2. Municipal recycling:

- Deletion results:
    - population_density: significant negative effect (-14.92, t = -2.99, p = 0.003) → denser populations have lower municipal recycling
    - Other variables (gdp_per_capita, education_attainment, waste_expenditure): not significant

- Imputation results:
    - population_density: remains significant 
    - Other predictors: still not significant.

Population density consistently affects municipal recycling, regardless of missing value handling.


3. Electrical recycling: 

- Deletion results:
    - population_density: strongly negative and highly significant (b = -19.89, t = -6.77, p < 0.001) → denser populations recycle less electrical waste
    - waste_expenditure: slightly significant negative effect (b = -5.12, t = -2.07, p = 0.039)
    - gdp_per_capita and education_attainment: not significant

- Imputation results:
    - population_density: still significant 
    - Other predictors: not significant

Population density is the strongest and most robust predictor for electrical recycling.

Overall Patterns: 

- Population density consistently shows negative effects for municipal and electrical recycling; its effect on packaging recycling is only significant in the deletion case.

- GDP and education are generally not significant once imputation is applied.

- Waste expenditure shows weak or inconsistent effects.



### Principal Component Analysis (PCA)
PCA is chosen due to it's dimension reduction, allowing for visualization of the similarities between the countries. Since there is correlation between the variables, this method should give some exploratory insights to see which variables are correlated, and what seperates the different country characteristics. 

In contrast to the regression above, this analysis looks at countries characteristics as a whole, also in aggregated form. This aims to compare between different countries, rather than focusing on influences within countries.

Since there are many missing values for both battery and vehicle recycling, these variables are dropped and the analysis focuses on the three main parts of the above analysis (Questions 1-3) as well: municipal, packaging and e-waste recycling. Furthermore, all rows with missing data are dropped. While this leads considerable information loss, since this is an exploratory anaylisis using a method that cannot handle missing values, it is not deemed a major issue. It is important to note here, that this can introduce potential bias towards countries that have more complete data.

#### PCA keeping time dimension unchanged
Unsure of how to deal with the time-series nature of the data in this analysis, several approaches are explored. First, the year variable is introduced as its own variable in the PCA. This results in multiple scores for the same country, but it also allows the influence of time to be shown in the components.

In [ ]:
merged_data_PCA = df_merged.copy()

merged_data_PCA = merged_data_PCA.drop(["battery_recycling", "vehicle_recycling", "joining_year"], axis = 1)
features = merged_data_PCA.columns.tolist
merged_data_PCA = merged_data_PCA.dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(merged_data_PCA.drop("country", axis=1))

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_

pca_summary = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained_var))],
    "explained_variance": explained_var,
    "cumulative_variance": np.cumsum(explained_var)
})

pca_summary

The first two principal components explain around 50% of the total variance - approximately six principal components would be necessary to explain 90% of the variance. Still, the first two PCs allow for a meaningful visualization of the data.

In [ ]:
numeric_features = [
    "year",
    "packaging_recycling",
    "municipal_recycling",
    "electrical_recycling",
    "gdp_per_capita",
    "urban_population_share",
    "population_density",
    "education_attainment",
    "waste_expenditure"
]

loadings = pd.DataFrame(
    pca.components_.T,
    index=numeric_features,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

loadings

Looking at the loadings can give a hint as to what contributes to each principal component. The first principal component is most strongly influenced by packaging and municipal recycling, GDP/capita and education attainment. In contrast, the urban population share, population density and waste expenditure of the countries contribute most strongly to the second principal component. 

In [ ]:
def pca_biplot(X_pca, loadings, labels, scale=3):
    
    plt.figure(figsize=(8, 6))

    # Scores
    plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7)

    for i, label in enumerate(labels):
        plt.text(
            X_pca[i, 0],
            X_pca[i, 1],
            label,
            fontsize=7,
            alpha=0.7
        )

    # Loadings
    for var in loadings.index:
        x = loadings.loc[var, "PC1"] * scale
        y = loadings.loc[var, "PC2"] * scale

        plt.arrow(
            0, 0, x, y,
            color="red",
            alpha=0.7,
            head_width=0.05,
            length_includes_head=True
        )
        plt.text(
            x * 1.1,
            y * 1.1,
            var,
            color="red",
            fontsize=9
        )

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.axhline(0, linewidth=0.5)
    plt.axvline(0, linewidth=0.5)
    plt.tight_layout()
    plt.show()


pca_biplot(
    X_pca,
    loadings,
    merged_data_PCA["country"].values
)

The visualization of the first two principal components can be seen in the biplot. Countries are represented by multiple scores, as the year variable is not explicitly handled, so each point represents the countries in a certain year. From the plot one can read that GDP/capita and education attainment are positively correlated with packaging and municipal recycling. Population density and waste expenditure are also positively correlated. 

No clear clusters emerge from the plot with the exception of Malta (MT), which differs strongly from the other countries along the second principal coponent. This suggests differences in population density, urban population share and waste expenditure. A small cluster could also be seen in the Netherlands (NL), Belgium (BE) and Luxembourg (LU), however it is not as clear as with Malta.

#### PCA aggregated over every 5 years
In order to reduce the time dimension, the data are aggregated into five-year periods for each country. As this approach still results in a number of missing values, rows with incomplete data are dropped prior to the PCA. This is argued to be finein this context, as the analysis is exploratory and aims to find similarities between countries that have high recycling numbers. 

In [ ]:
merged_data_PCA = df_merged.copy()

merged_data_PCA["period_start"] = 2000 + ((merged_data_PCA["year"] - 2000)//5) *5
merged_data_PCA["period_end"] = merged_data_PCA["period_start"] + 4
merged_data_PCA["country_period"] = (
    merged_data_PCA["country"] + "_" +
    merged_data_PCA["period_end"].astype(str)
) # only attach period end for readability in plot

numeric_features = [
    "packaging_recycling",
    "municipal_recycling",
    "electrical_recycling",
    "battery_recycling",
    "vehicle_recycling",
    "gdp_per_capita",
    "urban_population_share",
    "population_density",
    "education_attainment",
    "waste_expenditure"
]

merged_data_PCA = (
    merged_data_PCA
    .groupby("country_period", as_index=False)[numeric_features]
    .mean()
)

merged_data_PCA = merged_data_PCA.drop(["vehicle_recycling", "battery_recycling"], axis=1)
merged_data_PCA = merged_data_PCA.dropna()

features = [
    "packaging_recycling",
    "municipal_recycling",
    "electrical_recycling",
    "gdp_per_capita",
    "urban_population_share",
    "population_density",
    "education_attainment",
    "waste_expenditure"
]
X = merged_data_PCA.loc[:,features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_

pca_summary = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained_var))],
    "explained_variance": explained_var,
    "cumulative_variance": np.cumsum(explained_var)
})

pca_summary

Aggregating over the years leads to the first two principal components being able to account for almost 60% of the explained variance, now only 5 PCs are required to explain 90% of the variability. 

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

loadings

The loadings present a similar picture as to the previous analysis. The first principal component is again influenced mostly by packaging and municipal recycling, GDP/capita and education attainment. Again, the second principal component is impacted mostly by urban population share, population densityand waste expenditure, but also shows a notable contribution from electrical recycling.

In [ ]:
pca_biplot(
    X_pca,
    loadings,
    merged_data_PCA["country_period"].values
)

The five year aggregated data yield a biplot that is broadly comparable to the non-aggregated version. Here, Malta (MT), Romania (RO), Slowakia (SK) and Portugal (PT) in the early periods (2010-2014), Hungary (HU), Latvia (LV) and Croatia (HR) make up the lower end of PC1, indicating lower municipal and packaging recycling rates, as well as lower GDP per capita and education attainment. These counrties are also on the lower side of PC2, hinting at lower population density, waste expenditure and urban population share. In contrast, Luxembourg (LU), Belgium (BE), as well as the Netherlands (NL) and Ireland (IE) in the later periods (2020-2024) positioned at the higher end of PC1, while differing more strongly along PC2. These countries are characterized by higher GDP/capita and education attainment, along side higher municipal and packaging recycling rates.

Nevertheless, the first two principal components still account for only around 60% of the cumulative explained variance, and the interpretation of these patterns should therefore be treated with appropriate caution.

#### PCA aggregated over all years
Finally, this data is also considered without an explicit time dimension by aggregating over the years 2000-2024. This is a large time span, however it seems appropriate to consider country-level averages in this conrtext as well. This should also increase the number of countries included in the PCA, as fewer observations are lost due to missing values. Additionally, this will get rid of multiple scores per country in the biplot, potentially making the interpretation clearer. 

In [ ]:
merged_data_PCA = df_merged.copy()

merged_data_PCA = (
    merged_data_PCA
    .groupby("country", as_index=False)[numeric_features]
    .mean()
)

merged_data_PCA = merged_data_PCA.drop(["vehicle_recycling", "battery_recycling"], axis=1)
merged_data_PCA = merged_data_PCA.dropna()
merged_data_PCA

In [ ]:
X = merged_data_PCA.loc[:,features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_

pca_summary = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained_var))],
    "explained_variance": explained_var,
    "cumulative_variance": np.cumsum(explained_var)
})

pca_summary

When considering the countries over the entire time span the first two principal components already make up for almost 70% of the cumulative explained variance.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

loadings

Looking at the loadings can give a hint as to what contributes to each principal component. The first principal component is influenced most strongly by packaging and municipal recycling, GDP/capita and education attainment. In comparison the urban population share, population density and waste expenditure of the countries influence the second principal component most. 

In [ ]:
def pca_biplot(X_pca, loadings, labels, scale=3):
    
    plt.figure(figsize=(8, 6))

    # Scores
    plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7)

    for i, label in enumerate(labels):
        plt.text(
            X_pca[i, 0],
            X_pca[i, 1],
            label,
            fontsize=7,
            alpha=0.7
        )

    # Loadings
    for var in loadings.index:
        x = loadings.loc[var, "PC1"] * scale
        y = loadings.loc[var, "PC2"] * scale

        plt.arrow(
            0, 0, x, y,
            color="red",
            alpha=0.7,
            head_width=0.05,
            length_includes_head=True
        )
        plt.text(
            x * 1.1,
            y * 1.1,
            var,
            color="red",
            fontsize=9
        )

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.axhline(0, linewidth=0.5)
    plt.axvline(0, linewidth=0.5)
    plt.tight_layout()
    plt.show()


pca_biplot(
    X_pca,
    loadings,
    merged_data_PCA["country"].values
)

Here we can see the visualization of the first two principal components in a Bi-plot. Countries are represented by multiple scores, as the year variable is not handled, so each point represents the countries in a certain year. From the plot one can read that GDP/capita and education attainment are positively correlated with packaging and municipal recycling. Population density and waste expenditure are also positively correlated. 

No clear clusters emerge from the plot with the exception of MT (Malta) which differs strongly from the other group along PC2, so it differs in population density, urban population share and waste expenditure from the others. A small cluster could also be seen in the Netherlands (NL), Belgium (BE) and Luxemburg (LU), however it is not as clear as with Malta.

#### PCA aggregated over every 5 years
In order to reduce the time dimensions, all the countries are aggregated over every five years. Since this still produces quite a few missing values which is a problem for PCA, the rows with missing values are dropped. This is argued to be fine, since this is an exploratory attempt to find similarities between countries that have high recycling numbers. 

In [ ]:
merged_data_PCA = df_merged.copy()

merged_data_PCA["period_start"] = 2000 + ((merged_data_PCA["year"] - 2000)//5) *5
merged_data_PCA["period_end"] = merged_data_PCA["period_start"] + 4
merged_data_PCA["country_period"] = (
    merged_data_PCA["country"] + "_" +
    merged_data_PCA["period_end"].astype(str)
) # only attach period end for readability in plot

numeric_features = [
    "packaging_recycling",
    "municipal_recycling",
    "electrical_recycling",
    "battery_recycling",
    "vehicle_recycling",
    "gdp_per_capita",
    "urban_population_share",
    "population_density",
    "education_attainment",
    "waste_expenditure"
]

merged_data_PCA = (
    merged_data_PCA
    .groupby("country_period", as_index=False)[numeric_features]
    .mean()
)

merged_data_PCA = merged_data_PCA.drop(["vehicle_recycling", "battery_recycling"], axis=1)
merged_data_PCA = merged_data_PCA.dropna()

features = [
    "packaging_recycling",
    "municipal_recycling",
    "electrical_recycling",
    "gdp_per_capita",
    "urban_population_share",
    "population_density",
    "education_attainment",
    "waste_expenditure"
]
X = merged_data_PCA.loc[:,features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_

pca_summary = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained_var))],
    "explained_variance": explained_var,
    "cumulative_variance": np.cumsum(explained_var)
})

pca_summary

Aggregating over the years has lead to the first two principal components being able to account for almost 60% of the explained variance, now only 5 PCs would be needed in order to explain 90% of the variability. 

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

loadings

The loadings paint a similar picture as before, PC1 is influenced mostly by packaging and municipal recycling, GDP/capita and education attainment. Again, PC2 is influenced by urban population share, population density, waste expenditure but also electrical recycling. 

In [ ]:
pca_biplot(
    X_pca,
    loadings,
    merged_data_PCA["country_period"].values
)

The data aggregated over 5 years shows a similar Bi-plot as before for the data without the aggregation. Here, Malta, Romania (RO), Poland (PO), Hungary (HU), Latvia (LV) and Kroatia (HR) make up the lower end of PC1, meaning lower municipal and packaging recycling rates, but also lower GDP/capita, education attainment. These counrties are also on the lower side of PC2, hinting at lower population density, waste expenditure and urban population share. Luxemburg (LU), Belgium (BE), as well as the Netherlands (NL) and Ireland (IE) in the later years (2020-2024) are on the higher end of PC1, while differing quite a bit along PC2. These countries have a higher GDP/capita and education attainment, as well as higher municipal and packaging recycling rates. 

Again, the first principal components only make up around 60% of the cumulative explained variance, so these findings should be weighted accordingly. 

#### PCA aggregated over all years
Finally, this data is also considered without the time element, aggregating over the years 2000-2024. This is a large time span, however it seems appropriate to consider the averages as well - this should also increase the number of countries considered in the PCA, as less will fall away due to missing values. This will get rid of the multiple country scores in the plot, perhaps making the interpretation clearer. 

In [ ]:
merged_data_PCA = df_merged.copy()

merged_data_PCA = (
    merged_data_PCA
    .groupby("country", as_index=False)[numeric_features]
    .mean()
)

merged_data_PCA = merged_data_PCA.drop(["vehicle_recycling", "battery_recycling"], axis=1)
merged_data_PCA = merged_data_PCA.dropna()
merged_data_PCA

In [ ]:
X = merged_data_PCA.loc[:,features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_

pca_summary = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained_var))],
    "explained_variance": explained_var,
    "cumulative_variance": np.cumsum(explained_var)
})

pca_summary

When considering the countries over the entire time span, the first two principal components account for almost 70% of the cumulative variance.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

loadings

The interpretation of the loadings for the first two principal components stay largely unchanged. However, for the first principal component, the variables that had less influence in the previous cases now influence it more strongly. For the second principal component, urban population share and population density stay the most influencial variables, with electrical recycling and waste expenditure also playing an important role.

In [ ]:
pca_biplot(
    X_pca,
    loadings,
    merged_data_PCA["country"].values
)

This final biplot suggest the presence of two groups (seperated mostly by PC1), though they are not clearly seperable, as well as a definite outlier, Malta (MT). The Netherlands (NL), Belgium (BE) and Luxembourg (LU) show one end of the spectrum, characterized by high GDP per capita, education attainment and municipal and packaging recycling rates, as well as a higher urban poplation share. On the other side of the spectrum are Romania (RO), Hungary (HU), Poland (PL), Slovakia (SK) and Kroatia (HR). Overall, the PCA allows for a nice visualization of the different characteristics across the countries and suggests that higher GDP/capita, education attainment and urban population share are positively correlated with municipal and packaging recycling rates, while population density and waste expenditure are negatively correlated with electrical recycling. 

This final biplot suggests the presence of two broad groups, separated primarily along the first principal component, although these groups are not clearly separable, as well as one distinct outlier, Malta (MT). The Netherlands (NL), Belgium (BE), and Luxembourg (LU) occupy one end of the spectrum, characterized by high GDP per capita, educational attainment, and municipal and packaging recycling rates, as well as a higher urban population share. On the opposite end are Romania (RO), Hungary (HU), Poland (PL), and Croatia (HR). Overall, the PCA allows for a clear visualization of cross-country differences and suggests that higher GDP per capita, educational attainment, and urban population share are positively correlated with municipal and packaging recycling rates, while population density and waste expenditure appear to be negatively correlated with electrical recycling.

### Summary
* GDP/capita, education attainment and urban population share are positively correlated to municipal and packaging recycling.
* PCA seperates countries along the first principal component, largely through GDP/capita, education attainment, municipal and packaging recycling and urban population share. Along the second principal component countries are seperated largely by population density, urban population share, waste expenditure and electrical recycling.
* PCA highlights cross-country heterogeneity, with higher-income and more urbanized countries tending to exhibit higher recycling rates.
* Malta differs highly from the other countries in its characteristics according to the PCA

### Hierarchical Clustering
Hierarchical clustering appears to be an appropriate method to use alongside PCA. While PCA provides a continuous representation of similarities between countries, clustering allows for the identification of discrete groups with broadly similar characteristics. This may help to support assumptions about which structural characteristics are associated with higher recycling rates. For this analysis, the data aggregated over all years are used in order to retain as many countries as possible.

The clustering is performed using Ward’s method on the scaled original variables.

In [ ]:
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(8, 6))
dendrogram(
    Z,
    labels=merged_data_PCA["country"].values,
    leaf_rotation=90
)
plt.tight_layout()
plt.show()

Upon first glance of the dendogram given from the hierarchical clustering shows three clusters. These broadly align with the patterns observed in the PCA, with Luxemburg (LU), the Netherlands (NL), Belgium (BE) etc. building one end of the spectrum and Romania (RO), Kroatia (HR) etc. building the other end of the spectrum. Seeing these clusters on the bi-plot (shown below) shows the similarity of the clusters found in the PCA analysis above.

In [ ]:
merged_data_PCA["cluster"] = fcluster(Z, t=3, criterion="maxclust")

cluster_counts = ( 
    merged_data_PCA 
    .groupby("cluster") 
    .size() 
    .rename("n_observations") 
)
print(cluster_counts)

cluster_summary = (
    merged_data_PCA
    .select_dtypes(include="number")
    .groupby("cluster")
    .agg(["mean", "std"])
)
cluster_summary

Examining the cluster-wise averages confirms the patterns suggested by the PCA. The two main clusters are of similar size, making a comparison of averages meaningful. Cluster 1, which includes Luxemburg, Belgium, Austria etc., sports higher average packaging recycling rates and approximately double the average municipal recycling rates compared to Cluster 2, which includes Romania, Croatia, Hungary etc. In terms of electrical recycling both clusters show similar averages. The countries in Cluster 1 also show almost double the average GDP/capita, as well as a higher average urban population share and education attainment. An unexpected result is the population density, which differs from what the PCA seemed to show, where the average is higher for Cluster 1 over Cluster 2. The standard deviations show that there is substancial variety and heterogeneity within the clusters as well.

The third cluster, which only contains Malta, differs quite heavily from the other groups - Malta shows significantly lower recycling rates across all three waste types. While its GDP/capita is not out of the ordinary for either clusters, it has a very high urban population share and population density likely due to its small land area. Another strong difference that destinguishes it from the other clusters is its waste expenditure, which is very high - this might be due it being a small island state, though this interpretation remains speculative.

In [ ]:
clusters = np.unique(merged_data_PCA["cluster"])
 
for cluster in clusters:
    mask = merged_data_PCA["cluster"] == cluster
    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        alpha=0.8,
        label=f"Cluster {cluster}"
    )
 
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
 
for i, label in enumerate(merged_data_PCA["country"]):
    plt.text(
        X_pca[i, 0],
        X_pca[i, 1],
        label,
        fontsize=7,
        alpha=0.7
    )
    
plt.tight_layout()
plt.show()

### Summary
* From hierarcical clustering three clusters emerge. Cluster 1 is characterized by higher average recycling rates alongside higher GDP per capita, population density, urban population share and educational attainment, as well as lower waste expenditure.
* Malta emerges as a distinct outlier, with comparatively low recycling rates and very high waste expenditure, population density, and urban population share. 

While PCA and hierarchical clustering highlight across-country patterns in recycling within the EU, these methods are exploratory and descriptive, sensitive to scaling, variable selection, and data aggregation, and do not establish causal relationships, but they provide a clear summary of structural factors linked to recycling performance.

# Question 5

## Read in dataset and modify it for regression analysis

I started by doing this analysis only for mucipipal waste.
I will later change it to other response variables, but to test out the diffent means this was a good first step.
I also dropped the policy variables and calculated a policy count instead as this lead to too much multicolliniarity issues.

In [ ]:
df = pd.read_csv("./data/merged_dataset_all.csv")
policy_years_cols = ['WFD', 'WEEE_Directive_1', 'WEEE_Directive_2', 'WEEE_Directive_3', 'WEEE_Directive_4', 'RoHS_1', 'RoHS_2', 
                    'BL', 'EOLVD', 'LFD_1', 'LFD_2', 'SUP', 'CEAP_1', 'CEAP_2', 'CEAP_3', 'DRS', 'PT']
df["policy_count"] = (df[policy_years_cols] > 0).sum(axis=1)
df = df.drop(policy_years_cols, axis = 1)
df = df[df['country'] != "EU27_2020"]
df['joining_year'] = df['joining_year'].astype('int64')
df = df.drop(['packaging_recycling', 'electrical_recycling', 'battery_recycling', 'vehicle_recycling'],axis = 1)
df = df.dropna(subset=["municipal_recycling"])
df = df.sort_values(["country", "year"]).copy()

## Calculate correlation

In [ ]:
corr = df.drop('country', axis = 1).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


We can see that the year and poicy count is highly correlated which is not surprising. As more years pass, more policies are added. I will see hoch this affects the regressoin model and remove the year ant one try.

## Simple linear regression

All years until 2018, are training years and I will try to predict the recycling rate for the later years.
I do start with a simple linear regression and I will build on that later.

### Creation of the models ###

In [ ]:
num_cols = [
    'gdp_per_capita',
    'urban_population_share',
    'population_density',
    'education_attainment',
    'waste_expenditure'
]

In [ ]:
### With country and year ###
df_cy = df.copy()

## Imputate missing values in training set ##
# Interpolations if some variables are missing in the middle of the time series
df_cy[num_cols] = df_cy.groupby("country")[num_cols].transform(
    lambda x: x.interpolate(method="linear", limit_area="inside")
)
#Forward and backwardsfill, if we have some small gaps at the beginning and the end
df_cy[num_cols] = df_cy.groupby("country")[num_cols].ffill(limit=3)
df_cy[num_cols] = df_cy.groupby("country")[num_cols].bfill(limit=3)
#Imputatation of -1 for the rest
for col in ['gdp_per_capita', 'urban_population_share', 'population_density', 'education_attainment', 'waste_expenditure']: 
    df_cy[col + "_missing"] = df_cy[col].isna().astype(int)
    df_cy[col] = df_cy[col].fillna(-1)
df_cy

df_cy = pd.get_dummies(
    df_cy,
    columns=['country'],
    drop_first=True,
    dtype=int 
)

train_cy = df_cy[df_cy['year'] <= 2018].copy()
test_cy  = df_cy[df_cy['year'] > 2018].copy()

X_train_cy = train_cy.drop("municipal_recycling", axis = 1)
y_train_cy = train_cy['municipal_recycling']

X_train_cy = sm.add_constant(X_train_cy)
model_cy = sm.OLS(y_train_cy, X_train_cy)
results_cy = model_cy.fit()
print(results_cy.summary())



In [ ]:
### Without country ###
df_y = df_cy.drop(
    columns=[c for c in df_cy.columns if c.startswith("country")]
)

train_y = df_y[df_y['year'] <= 2018].copy()
test_y  = df_y[df_y['year'] > 2018].copy()

import statsmodels.api as sm
X_train_y = train_y.drop("municipal_recycling", axis = 1)
y_train_y = train_y['municipal_recycling']

X_train_y = sm.add_constant(X_train_y)
model_y = sm.OLS(y_train_y, X_train_y)
results_y1 = model_y.fit()
print(results_y1.summary())

In [ ]:
### Without country and year ###
train_nn = df_y[df_y['year'] <= 2018].copy()
test_nn  = df_y[df_y['year'] > 2018].copy()

train_nn = train_nn.drop('year', axis = 1)
test_nn = test_nn.drop('year', axis = 1)

import statsmodels.api as sm
X_train_nn = train_nn.drop("municipal_recycling", axis = 1)
y_train_nn = train_nn['municipal_recycling']

X_train_nn = sm.add_constant(X_train_nn)
model_nn = sm.OLS(y_train_nn, X_train_nn)
results_nn = model_nn.fit()
print(results_nn.summary())

### Result comparison ###

In [ ]:
def result_analysis(X_train, test_set, resonse_var, results): 
    # Split X_test and y_test

    for col in ['gdp_per_capita', 'urban_population_share', 'population_density', 'education_attainment', 'waste_expenditure']: 
        test_set[col + "_missing"] = test_set[col].isna().astype(int)
        test_set[col] = test_set[col].fillna(-1)

    X_test = test_set.drop(resonse_var, axis=1)
    y_test = test_set[resonse_var]

    # Add constant and make sure columns align (Important for one-hot-encoding)
    X_test = sm.add_constant(X_test)
    X_test = X_test.reindex(
        columns=X_train.columns,
        fill_value=0
    )

    # Fill NA-values in test set --> I want to change that, there has to be a better way.
    X_test = X_test.fillna(-1)  
    y_pred = results.predict(X_test)

    # Calculate rmse, mae and r2
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE:  {mae:.2f}")
    print(f"R²:   {r2:.3f}")
    metrics = pd.DataFrame({
        "RMSE": [rmse],
        "MAE": [mae],
        "R2": [r2]
    })
 
    # Plot predicted vs. true values and the residuals.
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))  # each plot will be 6x6 approx

    # ---- Plot 1: Observed vs Predicted ----
    axes[0].scatter(y_test, y_pred, alpha=0.6)
    axes[0].plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        color="red",
        linestyle="--"
    )
    axes[0].set_xlabel("Observed municipal recycling")
    axes[0].set_ylabel("Predicted municipal recycling")
    axes[0].set_title("Observed vs Predicted")

    # ---- Plot 2: Residuals vs Fitted ----
    residuals = y_test - y_pred
    axes[1].scatter(y_pred, residuals, alpha=0.6)
    axes[1].axhline(0, color="red", linestyle="--")
    axes[1].set_xlabel("Predicted values")
    axes[1].set_ylabel("Residuals")
    axes[1].set_title("Residuals vs Fitted")

    plt.tight_layout()
    plt.show()
    return metrics
    


In [ ]:
pred_cy = result_analysis(X_train = X_train_cy, test_set = train_cy, resonse_var="municipal_recycling", results = results_cy)
pred_cy = result_analysis(X_train = X_train_cy, test_set = test_cy, resonse_var="municipal_recycling", results = results_cy)

In [ ]:
#pred_y = result_analysis(X_train = X_train_y, test_set = train_y, resonse_var="municipal_recycling", results = results_y)
#pred_cy = result_analysis(X_train = X_train_y, test_set = test_y, resonse_var="municipal_recycling", results = results_y)

In [ ]:
pred_y = result_analysis(X_train = X_train_nn, test_set = train_nn, resonse_var="municipal_recycling", results = results_nn)
pred_cy = result_analysis(X_train = X_train_nn, test_set = test_nn, resonse_var="municipal_recycling", results = results_nn)

## Panel regression ##

In [ ]:
df_cy = df.copy()

## Imputate missing values in training set ##
# Interpolations if some variables are missing in the middle of the time series
df_cy[num_cols] = df_cy.groupby("country")[num_cols].transform(
    lambda x: x.interpolate(method="linear", limit_area="inside")
)
#Forward and backwardsfill, if we have some small gaps at the beginning and the end
df_cy[num_cols] = df_cy.groupby("country")[num_cols].ffill(limit=3)
df_cy[num_cols] = df_cy.groupby("country")[num_cols].bfill(limit=3)
#Imputatation of -1 for the rest
for col in ['gdp_per_capita', 'urban_population_share', 'population_density', 'education_attainment', 'waste_expenditure']: 
    df_cy[col + "_missing"] = df_cy[col].isna().astype(int)
    df_cy[col] = df_cy[col].fillna(-1)
df_cy
df_cy = df_cy.set_index(['country', 'year']).sort_index()

train_cy = df_cy[df_cy.index.get_level_values('year') <= 2018].copy()
test_cy  = df_cy[df_cy.index.get_level_values('year') > 2018].copy()


X_train_cy = train_cy.drop("municipal_recycling", axis = 1)
y_train_cy = train_cy['municipal_recycling']

exog = sm.tools.tools.add_constant(X_train_cy)
endog = y_train_cy
mod = PooledOLS(endog, exog, check_rank=False)
pooledOLS_res = mod.fit()
# Store values for checking homoskedasticity graphically
#fittedvals_pooled_OLS = pooledOLS_res.predict().fitted_values
#residuals_pooled_OLS = pooledOLS_res.resids

In [ ]:
def panel_prediction_analysis(
    results,
    train_df,
    test_df,
    target,
    add_constant=True
):

    X_train = train_df.drop(columns=target)
    y_train = train_df[target]

    if add_constant:
        X_train = sm.add_constant(X_train, has_constant="add")

    y_train_pred = results.predict(X_train)

    # --------------------
    # Prepare TEST data
    # --------------------
    X_test = test_df.drop(columns=target)
    y_test = test_df[target]

    if add_constant:
        X_test = sm.add_constant(X_test, has_constant="add")

    # Ensure column alignment
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    y_test_pred = results.predict(X_test)

    # --------------------
    # Metrics
    # --------------------
    metrics = {
        "Train": {
            "RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
            "MAE": mean_absolute_error(y_train, y_train_pred),
            "R2": r2_score(y_train, y_train_pred),
        },
        "Test": {
            "RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
            "MAE": mean_absolute_error(y_test, y_test_pred),
            "R2": r2_score(y_test, y_test_pred),
        },
    }
    print(metrics)

    # --------------------
    # Plotting
    # --------------------
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

    # Train plot
    axes[0].scatter(y_train, y_train_pred, alpha=0.6)
    axes[0].plot(
        [y_train.min(), y_train.max()],
        [y_train.min(), y_train.max()],
        linestyle="--"
    )
    axes[0].set_title("Train: Observed vs Predicted")
    axes[0].set_xlabel("Observed")
    axes[0].set_ylabel("Predicted")

    # Test plot
    axes[1].scatter(y_test, y_test_pred, alpha=0.6)
    axes[1].plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        linestyle="--"
    )
    axes[1].set_title("Test: Observed vs Predicted")
    axes[1].set_xlabel("Observed")

    plt.tight_layout()
    plt.show()

    return metrics


In [ ]:
metrics = panel_prediction_analysis(
    results=pooledOLS_res,
    train_df=train_cy,
    test_df=test_cy,
    target="municipal_recycling",
    add_constant=True
)

print(metrics)


In [ ]:
from linearmodels.panel import PanelOLS

df_cy = df.copy()

## Imputate missing values in training set ##
# Interpolations if some variables are missing in the middle of the time series
df_cy[num_cols] = df_cy.groupby("country")[num_cols].transform(
    lambda x: x.interpolate(method="linear", limit_area="inside")
)
#Forward and backwardsfill, if we have some small gaps at the beginning and the end
df_cy[num_cols] = df_cy.groupby("country")[num_cols].ffill(limit=3)
df_cy[num_cols] = df_cy.groupby("country")[num_cols].bfill(limit=3)
#Imputatation of -1 for the rest
for col in ['gdp_per_capita', 'urban_population_share', 'population_density', 'education_attainment', 'waste_expenditure']: 
    df_cy[col + "_missing"] = df_cy[col].isna().astype(int)
    df_cy[col] = df_cy[col].fillna(-1)
df_cy
df_cy = df_cy.set_index(['country', 'year']).sort_index()

train_cy = df_cy[df_cy.index.get_level_values('year') <= 2018].copy()
test_cy  = df_cy[df_cy.index.get_level_values('year') > 2018].copy()


X_train_cy = train_cy.drop("municipal_recycling", axis = 1)
y_train_cy = train_cy['municipal_recycling']

#X_train_cy = sm.add_constant(X_train_cy)

# Fixed effects model
model = PanelOLS(y_train_cy, X_train_cy, entity_effects=True, time_effects=True, check_rank=False, drop_absorbed=True)
results = model.fit()
print(results.summary)


In [ ]:
def panel_fe_prediction_analysis(
    results,
    train_df,
    test_df,
    target
):
    import matplotlib.pyplot as plt
    import numpy as np
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    used_cols = results.model.exog.vars

    # TRAIN
    X_train = train_df[used_cols]
    y_train = train_df[target]
    y_train_pred = results.predict(X_train)

    # TEST
    X_test = test_df[used_cols]
    y_test = test_df[target]
    y_test_pred = results.predict(X_test)

    metrics = {
        "Train": {
            "RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
            "MAE": mean_absolute_error(y_train, y_train_pred),
            "R2": r2_score(y_train, y_train_pred),
        },
        "Test": {
            "RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
            "MAE": mean_absolute_error(y_test, y_test_pred),
            "R2": r2_score(y_test, y_test_pred),
        },
    }

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

    axes[0].scatter(y_train, y_train_pred, alpha=0.6)
    axes[0].plot(
        [y_train.min(), y_train.max()],
        [y_train.min(), y_train.max()],
        linestyle="--"
    )
    axes[0].set_title("Train: Observed vs Predicted")
    axes[0].set_xlabel("Observed")
    axes[0].set_ylabel("Predicted")

    axes[1].scatter(y_test, y_test_pred, alpha=0.6)
    axes[1].plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        linestyle="--"
    )
    axes[1].set_title("Test: Observed vs Predicted")
    axes[1].set_xlabel("Observed")

    plt.tight_layout()
    plt.show()

    return metrics


In [ ]:
metrics = panel_fe_prediction_analysis(
    results=results,
    train_df=train_cy,
    test_df=test_cy,
    target="municipal_recycling"
)

print(metrics)


## Time Series Models

Check if data is stationary

In [ ]:
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt

def check_stationarity(country, dataset, response_var, d=0):

    ### Select data ###
    df = dataset[dataset['country'] == country]
    mun_w = df[['year', response_var]]
    mun_w = mun_w.set_index('year')

    # Apply differencing if d > 0
    y = mun_w[response_var].copy()
    for _ in range(d):
        y = y.diff().dropna()

    ### Plot ###
    if d == 0:
        plt.figure(figsize=(4,3))
        plt.plot(y, marker='o')
        plt.ylim(0, 100)
        plt.xlabel('Year')
        plt.ylabel(f'{country} (%)')
        plt.title(f'Stationarity check (d={d})')
        plt.show()
    else:
        plt.figure(figsize=(4,3))
        plt.plot(y, marker='o')
        plt.ylim()
        plt.xlabel('Year')
        plt.ylabel(f'{country} (%)')
        plt.title(f'Stationarity check (d={d})')
        plt.show()

    ### Means & Variances ###
    values = y.values
    parts = int(len(values) / 3)

    part_1 = values[0:parts]
    part_2 = values[parts:2*parts]
    part_3 = values[2*parts:3*parts]

    mean_1, mean_2, mean_3 = part_1.mean(), part_2.mean(), part_3.mean()
    var_1, var_2, var_3 = part_1.var(), part_2.var(), part_3.var()

    print(f'mean1={mean_1:.4f}, mean2={mean_2:.4f}, mean3={mean_3:.4f}')
    print(f'var1={var_1:.4f}, var2={var_2:.4f}, var3={var_3:.4f}')

    ### Augmented Dickey-Fuller test ###
    res = adfuller(values)

    print(f'ADF Statistic: {res[0]:.6f}')
    print(f'p-value: {res[1]:.6f}')
    print('Critical values:')
    for k, v in res[4].items():
        print(f'  {k}: {v:.3f}')

    return mun_w


In [ ]:

def check_orders(y, x, year, response_var, p_range, q_range):
    
    warnings.filterwarnings("ignore", category=UserWarning)

    fig, axes = plt.subplots(1, 2, figsize=(9, 3))  

    plot_acf(y, ax=axes[0], lags=6)
    axes[0].set_title('ACF')

    plot_pacf(y, ax=axes[1], lags=6)
    axes[1].set_title('PACF')
    plt.tight_layout()
    plt.show()

    results = []
    for p, q in itertools.product(range(4), range(4)):
        try:
            model = ARIMA(y, order=(p, 0, q))
            res = model.fit()
            results.append({
                'p': p,
                'q': q,
                'AIC': res.aic,
                'BIC': res.bic
            })
        except:
            pass

    results_df = pd.DataFrame(results)
    print(results_df.sort_values('AIC', ascending = True).head(5))

In [ ]:
def arima_model(y, x, p, d, q):
    
    model = ARIMA(y, order=(p, d, q))
    model_fit = model.fit()

    forecast_res = model_fit.get_forecast(steps=5)
    forecast_mean = forecast_res.predicted_mean
    conf_int = forecast_res.conf_int()

    print(forecast_mean)
    print(conf_int)

    naive = y[y.index == '2018-01-01']
    naive = pd.Series(np.repeat(naive, len(forecast_mean)))
    naive.index = forecast_mean.index
    
    #naive_forecast = pd.Series(
    #    [train.iloc[-1]] * h,
    #    index=test.index
    #)

    rolling_forecast = []
    history = y.tolist()

    h = len(naive)     
    window = 3
    for _ in range(h):
        next_forecast = np.mean(history[-window:])
        rolling_forecast.append(next_forecast)
        history.append(next_forecast)

    rolling_forecast = pd.Series(rolling_forecast)
    rolling_forecast.index = forecast_mean.index

    plt.figure(figsize=(6, 4))
    plt.plot(y, label='Train')
    plt.plot(x, label='Test', color='black')
    plt.plot(forecast_mean, label='Forecast', linestyle='--')
    plt.plot(naive, label='Naive', linestyle='--')
    plt.plot(rolling_forecast, label='Rolling_mean', linestyle='--')

    plt.fill_between(
        conf_int.index,
        conf_int.iloc[:, 0],
        conf_int.iloc[:, 1],
        alpha=0.3
    )

    plt.ylim(0, 100)
    plt.legend()
    plt.title('Municipal recycling – forecast')
    plt.show()


    model_fit.plot_diagnostics(figsize=(10,6))



    rmse = np.sqrt(mean_squared_error(x, forecast_mean))
    mae = mean_absolute_error(x, forecast_mean)

    rmse_naive = np.sqrt(mean_squared_error(x, naive))
    mae_naive = mean_absolute_error(x, naive)

    rmse_rolling = np.sqrt(mean_squared_error(x, rolling_forecast))
    mae_rolling = mean_absolute_error(x, rolling_forecast)

    print(f"rmse ARIMA: {rmse}")
    print(f"rmse naive: {rmse_naive}")
    print(f"rmse rolling mean: {rmse_rolling}")

    print(f"mae ARIMA: {mae}")
    print(f"mae naive: {mae_naive}")
    print(f"mae rolling mean: {mae_rolling}")



### Try out AT for municipal waste ###

In [ ]:
df_AT = check_stationarity('AT', df, 'municipal_recycling', 0) # --> AT is stationary

In [ ]:
train = df_AT[df_AT.index.get_level_values('year') <= 2018].copy()
test  = df_AT[df_AT.index.get_level_values('year') > 2018].copy()

y = train['municipal_recycling']
y.index = pd.to_datetime(y.index, format='%Y')

x = test['municipal_recycling']
x.index = pd.to_datetime(x.index, format='%Y')

check_orders(y, x, 2018, 'municipal_recycling', 6, 6)

In [ ]:
arima_model(y,x, 1, 0, 0)

### Try out DE with municipal waste ###

In [ ]:
df_DE = check_stationarity('DE', df, 'municipal_recycling') # --> DE is not stationary
df_DE = check_stationarity('DE', df, 'municipal_recycling', 1)

In [ ]:
train = df_DE[df_DE.index.get_level_values('year') <= 2018].copy()
test  = df_DE[df_DE.index.get_level_values('year') > 2018].copy()

y = train['municipal_recycling']
y.index = pd.to_datetime(y.index, format='%Y')

x = test['municipal_recycling']
x.index = pd.to_datetime(x.index, format='%Y')

check_orders(y, x, 2018, 'municipal_recycling', 6, 6)

In [ ]:
arima_model(y,x, 1, 1, 1)

# 7. Findings

## 7.1. Potential bias

In question 4, the exclusion of observations with missing values may bias the analysis toward countries with more complete reporting, despite aggregation mitigating some data loss. While PCA and hierarchical clustering highlight across-country patterns in recycling within the EU, these methods are exploratory and descriptive. They are sensitive to scaling, variable selection, and data aggregation, and do not establish causal relationships - they do, however, provide a clear summary of structural factors linked to recycling performance.

## 7.2. Why were the questions good?

Q: Are there characteristics of countries that could lead to increased recycling?

After analyzing the development of recycling over time, this question investigates country-level characteristics to better understand the factors associated with lower or higher waste recycling rates. The insights gained from the analysis enable more informed and targeted decision-making regarding future regulations for increasing recycling across EU countries. 



## 7.3. Do the answers make any sense?



## 7.4. Were there any difficulties?
    -flags
    -policy changes
    -lot of difference datasets

The Eurostat data included flags indicating varying levels of quality or reliability. After consideration, the numbers were used as reported, since no alternative was available. In a future project, the flags could be incorporated more systematically to assess data quality.

Merging the different datasets proved challenging and required substantial effort. The datasets cover different countries and time periods, which led to many missing values in the combined dataset and required careful preprocessing to create a unified dataset suitable for analysis.
 

## 7.5. Which Data Science tools and techniques were learned during this exercise?

In Question 4 for the within-country exploratory analysis, a panel regression model was applied. For exploration across countries, Principal Component Analysis (PCA) and Hierarchical Clustering were used to visualize patterns, group countries with similar characteristics, and examine structural factors associated with recycling rates.

## 7.6. Key insights

* Question 4: Exploratory analyses using PCA and Hierarchical Clustering consistently highlight recycling patterns across countries within the EU. Higher GDP per capita, education attainment, and urbanization are associated with higher municipal and packaging recycling rates, while lower values correspond to lower recycling.

